# Agoda Data Engineering System Design — Complete Interview Prep

## Round 2: System Design (Data Engineering)

**What this notebook covers:**
1. Agoda Interview Format & Recently Asked Questions (2024–2026)
2. End-to-End System Design Framework for Data Engineering
3. OTA Platform Data Architecture — Deep Dive
4. Data Ingestion at Scale (1M+ QPS, TB-scale)
5. Data Processing: Batch & Real-Time (Spark, Kafka, Flink)
6. Data Modeling & Storage (SQL, NoSQL, Lakehouse)
7. Feature Store & ML Pipelines (Agoda's Featureflow)
8. Performance, Scalability & Optimization
9. Operational Excellence: Monitoring, Alerting, Recovery
10. Full System Design Walkthroughs (3 complete designs)
11. Common Mistakes, Tricks, Edge Cases
12. Mock Interview Q&A Bank

---

# Part 1: Agoda Interview Format & Recently Asked Questions

## 1.1 Interview Structure

Agoda's Data Engineering interview typically has these rounds:

| Round | Format | Duration | Focus |
|-------|--------|----------|-------|
| Screening | Recruiter call | 30 min | Experience, motivation, skill level |
| Round 1 | Coding (CodeSignal) | 60-90 min | DSA, SQL, Python (LeetCode easy-medium) |
| **Round 2** | **System Design (Data Eng)** | **45-60 min** | **Scalable data systems, pipelines, architecture** |
| Round 3 | Platform Design / Behavioral | 45-60 min | Review existing architecture, propose improvements |

## 1.2 What Makes Agoda's Round Unique

**Key Insight:** Agoda's system design round for data engineering is NOT a typical "design Twitter" interview. It focuses on:
- **Data pipeline design** (ingestion → processing → storage → serving)
- **Real-world OTA scenarios** (pricing, inventory, search ranking, bookings)
- **Reviewing & improving existing designs** (not always building from scratch)
- **Operational maturity** (monitoring, alerting, recovery, data quality)

## 1.3 Recently Asked Questions (2024–2026)

Based on interview experiences shared on Medium, Glassdoor, AmbitionBox, and InterviewQuery:

### System Design Questions
1. **Design a real-time hotel pricing pipeline** that processes rate updates from 2M+ properties and serves fresh prices to search results within 5 seconds
2. **Design a hotel search ranking data pipeline** — feature engineering, model training, online feature serving
3. **Design an event ticket booking system** — API design, database choices, scalability, bottlenecks
4. **Design a clickstream analytics pipeline** for tracking user behavior on agoda.com (1.8T events/day)
5. **Design a data quality monitoring system** for detecting anomalies in booking data
6. **Design a feature store** for ML models (online + offline serving, 3.5M entities/sec)
7. **Design a real-time inventory management system** preventing overbooking across multiple channels
8. **Design a unified data pipeline** consolidating multiple data sources into a single source of truth
9. **Design a CDC (Change Data Capture) pipeline** from MySQL to data lake for analytics
10. **Design a recommendation/personalization data pipeline** — user behavior → features → model → serving

### Platform Review Questions
11. Review an existing booking service expanding to support booking agents — identify gaps in concurrency, consistency, and security
12. Review a data pipeline architecture and propose improvements for fault tolerance and scalability

### Deep-Dive Follow-Up Questions
- How do you handle data skew in Spark jobs processing hotel data by city?
- How do you ensure exactly-once processing in your Kafka pipeline?
- What happens when your Spark job fails halfway through processing 500GB of data?
- How do you handle schema evolution when upstream teams change their data format?
- What's your strategy for backfilling 6 months of historical data after a pipeline bug fix?
- How do you handle late-arriving events in a streaming pipeline?

# Part 2: End-to-End System Design Framework

## 2.1 The 5-Step Framework (Use This in Every Interview)

```
┌─────────────────────────────────────────────────────────────┐
│  Step 1: CLARIFY (3-5 min)                                  │
│  → Data sources, volume, velocity, latency SLA, consumers   │
├─────────────────────────────────────────────────────────────┤
│  Step 2: ESTIMATE (2-3 min)                                 │
│  → Back-of-envelope: QPS, storage, bandwidth                │
├─────────────────────────────────────────────────────────────┤
│  Step 3: HIGH-LEVEL DESIGN (10-15 min)                      │
│  → [Sources] → [Ingestion] → [Processing] → [Storage] → [Serving] │
├─────────────────────────────────────────────────────────────┤
│  Step 4: DEEP DIVE (10-15 min)                              │
│  → Schema, partitioning, error handling, exactly-once, backfill │
├─────────────────────────────────────────────────────────────┤
│  Step 5: OPERATIONAL EXCELLENCE (5 min)                     │
│  → Monitoring, alerting, recovery, data quality, SLAs       │
└─────────────────────────────────────────────────────────────┘
```

## 2.2 Clarifying Questions Cheat Sheet

**Always ask these before designing:**

| Category | Questions |
|----------|----------|
| **Data Sources** | What are the sources? APIs, databases, logs, streams? |
| **Volume** | How much data per day/hour/second? Growth rate? |
| **Velocity** | Real-time (<1s), near-real-time (<5min), or batch (hours)? |
| **Variety** | Structured, semi-structured, unstructured? Schema stability? |
| **Consumers** | Who reads the data? Dashboards, ML models, APIs, analysts? |
| **Latency SLA** | End-to-end latency requirement? |
| **Consistency** | Strong consistency or eventual consistency acceptable? |
| **Compliance** | PII handling, GDPR, data retention policies? |
| **Failure tolerance** | What happens if pipeline is down for 1 hour? |

## 2.3 Back-of-Envelope Estimation Templates

### Quick Reference Numbers
```
1 day = 86,400 seconds ≈ 100K seconds (for estimation)
1 million QPS = 86.4 billion events/day
1 KB per event × 1M QPS = 1 GB/sec = 86 TB/day
1 KB per event × 10K QPS = 10 MB/sec = 864 GB/day
```

In [ ]:
# Back-of-Envelope Estimation Calculator
# Use this during interviews to quickly estimate resource needs

class BackOfEnvelopeEstimator:
    SECONDS_PER_DAY = 86_400
    
    @staticmethod
    def qps_to_daily(qps: int) -> str:
        daily = qps * BackOfEnvelopeEstimator.SECONDS_PER_DAY
        if daily >= 1e12:
            return f"{daily/1e12:.1f} trillion events/day"
        elif daily >= 1e9:
            return f"{daily/1e9:.1f} billion events/day"
        elif daily >= 1e6:
            return f"{daily/1e6:.1f} million events/day"
        return f"{daily:,.0f} events/day"
    
    @staticmethod
    def storage_estimate(qps: int, avg_event_size_bytes: int, 
                         retention_days: int, replication_factor: int = 3) -> dict:
        daily_bytes = qps * BackOfEnvelopeEstimator.SECONDS_PER_DAY * avg_event_size_bytes
        total_bytes = daily_bytes * retention_days * replication_factor
        return {
            "throughput_mb_sec": round(qps * avg_event_size_bytes / 1e6, 2),
            "daily_raw_tb": round(daily_bytes / 1e12, 2),
            "total_storage_tb": round(total_bytes / 1e12, 2),
            "retention_days": retention_days,
            "replication_factor": replication_factor
        }
    
    @staticmethod
    def kafka_partitions(qps: int, throughput_per_partition: int = 10_000) -> int:
        """Estimate minimum Kafka partitions needed.
        Rule of thumb: each partition handles ~10K msg/sec for well-tuned clusters."""
        return max(1, -(-qps // throughput_per_partition))  # ceiling division
    
    @staticmethod
    def spark_cores(qps: int, processing_time_ms: float = 1.0) -> int:
        """Estimate Spark cores for streaming workload.
        Each core can process ~1000 events/sec for typical transformations."""
        events_per_core_per_sec = 1000 / processing_time_ms * 1000
        return max(1, int(qps / events_per_core_per_sec * 1.5))  # 1.5x safety margin


# --- Example: Agoda Clickstream Pipeline ---
est = BackOfEnvelopeEstimator()

# Agoda processes 1.8 trillion events/day ≈ ~20.8M events/sec
qps = 20_800_000
avg_event_size = 500  # bytes (clickstream event)

print("=== Agoda Clickstream Pipeline Estimation ===")
print(f"QPS: {qps:,}")
print(f"Daily volume: {est.qps_to_daily(qps)}")
print()

storage = est.storage_estimate(qps, avg_event_size, retention_days=30, replication_factor=3)
print(f"Throughput: {storage['throughput_mb_sec']} MB/sec")
print(f"Daily raw: {storage['daily_raw_tb']} TB")
print(f"Total storage (30d, 3x replication): {storage['total_storage_tb']} TB")
print()

partitions = est.kafka_partitions(qps)
print(f"Min Kafka partitions needed: {partitions}")
print(f"Recommended (2x buffer): {partitions * 2}")
print()

cores = est.spark_cores(qps)
print(f"Estimated Spark cores: {cores}")

# Part 3: OTA Platform Data Architecture — Deep Dive

## 3.1 Agoda's Data Ecosystem (What You Should Know)

```
┌──────────────────────────────────────────────────────────────────────┐
│                        AGODA DATA ECOSYSTEM                         │
├──────────────────────────────────────────────────────────────────────┤
│                                                                      │
│  DATA SOURCES                                                        │
│  ┌─────────┐ ┌──────────┐ ┌───────────┐ ┌───────────┐              │
│  │ Booking  │ │ Search   │ │ Pricing   │ │ Supplier  │              │
│  │ Service  │ │ Clicks   │ │ Updates   │ │ Feeds     │              │
│  └────┬─────┘ └────┬─────┘ └─────┬─────┘ └─────┬─────┘              │
│       │             │             │             │                     │
│  ─────┴─────────────┴─────────────┴─────────────┴────────            │
│                           │                                          │
│  INGESTION LAYER          ▼                                          │
│  ┌──────────────────────────────────────────┐                        │
│  │          Apache Kafka (1.8T events/day)   │                        │
│  │  ┌────────┐ ┌────────┐ ┌────────┐        │                        │
│  │  │Cluster │ │Cluster │ │Cluster │  ...    │                        │
│  │  │Analytics│ │RT Pipe │ │ML Pipe │        │                        │
│  │  └────────┘ └────────┘ └────────┘        │                        │
│  └──────────────────────────────────────────┘                        │
│                           │                                          │
│  PROCESSING LAYER         ▼                                          │
│  ┌────────────────┐ ┌────────────────┐ ┌────────────────┐           │
│  │  Spark Batch   │ │ Spark          │ │ Flink          │           │
│  │  (ETL, reports)│ │ Streaming      │ │ (Real-time)    │           │
│  └────────┬───────┘ └───────┬────────┘ └───────┬────────┘           │
│           │                  │                   │                    │
│  STORAGE LAYER              ▼                                        │
│  ┌──────────────────────────────────────────────────────┐           │
│  │  ┌──────────┐ ┌──────────┐ ┌──────────┐ ┌────────┐  │           │
│  │  │ Data Lake │ │  Data    │ │ Feature  │ │ OLTP   │  │           │
│  │  │ (HDFS/S3) │ │Warehouse │ │  Store   │ │ (MySQL)│  │           │
│  │  │ (raw/     │ │(Hive/    │ │(ScyllaDB │ │        │  │           │
│  │  │ curated)  │ │ Presto)  │ │ +Drgnfly)│ │        │  │           │
│  │  └──────────┘ └──────────┘ └──────────┘ └────────┘  │           │
│  └──────────────────────────────────────────────────────┘           │
│                           │                                          │
│  SERVING LAYER            ▼                                          │
│  ┌──────────────┐ ┌──────────────┐ ┌──────────────┐                │
│  │ Elasticsearch │ │  REST APIs   │ │  Dashboards  │                │
│  │ (search)      │ │  (feature    │ │  (Grafana/   │                │
│  │               │ │   serving)   │ │   internal)  │                │
│  └──────────────┘ └──────────────┘ └──────────────┘                │
└──────────────────────────────────────────────────────────────────────┘
```

## 3.2 Agoda's Key Technical Facts (Reference During Interview)

| Metric | Value | Source |
|--------|-------|--------|
| Daily Kafka events | 1.8 trillion | Agoda Engineering Blog |
| Kafka growth rate | 2x year-over-year since 2015 | Agoda Engineering Blog |
| Feature Store QPS | 3.5M entities/sec | ScyllaDB case study |
| Feature Store P99 latency | <10ms | Agoda Engineering Blog |
| Feature Store cache (Dragonfly) P99 | <3ms | Agoda Engineering Blog |
| Feature Store scaling | 50x in 2 years | ScyllaDB case study |
| Featureflow features processed | 10,000+ | Agoda Engineering Blog |
| Data pipeline team size | ~30 people, 4 teams | Firebolt case study |
| Ingestion architecture | 2-step logging (disk → forwarder → Kafka) | Agoda Engineering Blog |
| Analytics latency (P99) | ~10 seconds (2-step) | ByteByteGo analysis |
| Critical path latency | Sub-second (direct Kafka write) | ByteByteGo analysis |

## 3.3 OTA-Specific Data Domains

Understanding these domains is critical for Agoda interviews:

### Domain 1: Hotel Inventory & Availability
- **Challenge**: 2M+ properties × multiple room types × 365 days = billions of availability cells
- **Real-time requirement**: Availability must reflect bookings within seconds
- **Overbooking risk**: Same room sold on Agoda, Booking.com, direct channel simultaneously
- **Data model**: Property → Room Type → Rate Plan → Inventory (date-level)

### Domain 2: Dynamic Pricing
- **Challenge**: Prices change based on demand, competition, time-to-check-in, seasonality
- **Volume**: Millions of price updates per minute from suppliers
- **Freshness**: Stale prices = lost revenue or margin compression
- **ML models**: Price optimization models need real-time features

### Domain 3: Search & Ranking
- **Challenge**: Rank properties by relevance, value, conversion probability
- **Features needed**: Static (hotel attributes), slow-changing (reviews), real-time (price, availability)
- **Latency**: Search results must return in <500ms including ranking

### Domain 4: Booking & Payment
- **Challenge**: ACID transactions, idempotency, multi-currency
- **Consistency**: Cannot double-book; financial data must be exactly correct
- **Auditing**: Every transaction must be traceable end-to-end

### Domain 5: User Behavior & Personalization
- **Challenge**: Track and process clickstream data for personalization
- **Volume**: 1.8 trillion events/day
- **Use cases**: Recommendations, A/B testing, conversion optimization

# Part 4: Data Ingestion at Scale

## 4.1 Agoda's 2-Step Logging Architecture

This is one of Agoda's most important architectural decisions:

```
┌──────────────────┐     ┌──────────────────┐     ┌──────────────────┐
│  Application     │     │   Forwarder      │     │   Kafka          │
│  (writes to      │────▶│   Daemon         │────▶│   Cluster        │
│   local disk)    │     │   (reads files,  │     │                  │
│                  │     │    sends to Kafka)│     │                  │
└──────────────────┘     └──────────────────┘     └──────────────────┘

Step 1: App writes event      Step 2: Forwarder picks up
        to local disk                  and sends to Kafka
        (fast, no network)             (async, batched)
```

### Why This Design?

| Advantage | Explanation |
|-----------|------------|
| **Decoupled failure domains** | App doesn't fail if Kafka is down; events buffer on disk |
| **Low app-side latency** | Writing to local disk is microseconds vs. network call |
| **Independent upgrades** | Kafka team can upgrade forwarder without changing app code |
| **Backpressure handling** | Disk acts as buffer during traffic spikes |

### Trade-offs

| Trade-off | Impact |
|-----------|--------|
| **Higher latency** | P99 ~10 seconds for analytics (acceptable for non-critical) |
| **Disk space management** | Need to monitor disk usage; can fill up during prolonged Kafka outage |
| **At-least-once delivery** | Forwarder may re-send if it crashes after send but before ack |

### When to Bypass (Direct Kafka Write)
- Real-time pricing updates (sub-second latency needed)
- Booking confirmations (mission-critical, can't lose)
- Fraud detection signals (time-sensitive)

## 4.2 Kafka Architecture at Scale

In [ ]:
# Kafka Configuration Patterns for 1M+ QPS
# Understanding these configs is crucial for the interview

kafka_cluster_design = {
    "cluster_segmentation": {
        "principle": "Separate clusters by use case, NOT one giant cluster",
        "agoda_clusters": [
            "analytics-cluster (clickstream, page views)",
            "rt-pipeline-cluster (pricing, availability)",
            "ml-pipeline-cluster (feature computation, training data)",
            "dc-replication-cluster (cross data-center sync)",
            "async-api-cluster (async request/response)"
        ],
        "why": "Blast radius containment — issue in analytics cluster doesn't affect pricing"
    },
    
    "topic_design": {
        "partitioning_strategy": {
            "clickstream": "partition by user_id (ensures user event ordering)",
            "pricing": "partition by property_id (all prices for a hotel go to same partition)",
            "bookings": "partition by booking_id (ensures booking event ordering)",
            "inventory": "partition by property_id + room_type_id"
        },
        "partition_count": {
            "guideline": "Target 10-50 MB/sec per partition",
            "high_throughput_topic": "500-2000 partitions",
            "low_throughput_topic": "10-50 partitions"
        }
    },
    
    "producer_configs_for_high_throughput": {
        "batch.size": "64KB-256KB (larger batches = higher throughput)",
        "linger.ms": "5-20ms (wait to fill batch, increases latency slightly)",
        "compression.type": "lz4 or zstd (lz4 for speed, zstd for ratio)",
        "acks": "'all' for critical data, '1' for analytics",
        "buffer.memory": "64MB-256MB (total memory for batching)",
        "max.in.flight.requests.per.connection": "5 (with idempotent producer)"
    },
    
    "consumer_configs_for_high_throughput": {
        "fetch.min.bytes": "1MB (don't fetch tiny batches)",
        "fetch.max.wait.ms": "500ms (max wait for batch to fill)",
        "max.poll.records": "1000-5000 (records per poll)",
        "auto.offset.reset": "'earliest' for pipelines, 'latest' for monitoring"
    }
}

# Print the design document
import json
print(json.dumps(kafka_cluster_design, indent=2))

## 4.3 Data Ingestion Patterns

### Pattern 1: Event Sourcing (for bookings, transactions)
```
Event: {booking_id: 123, type: "CREATED", timestamp: ...}
Event: {booking_id: 123, type: "PAYMENT_RECEIVED", timestamp: ...}
Event: {booking_id: 123, type: "CONFIRMED", timestamp: ...}

Current state = replay all events for booking_id 123
```

### Pattern 2: CDC (Change Data Capture) — for database → data lake
```
MySQL binlog → Debezium → Kafka → Spark/Flink → Data Lake

Advantages:
- No changes to application code
- Captures every change (inserts, updates, deletes)
- Low latency (seconds, not hours like batch ETL)
```

### Pattern 3: API Polling (for supplier data)
```
Scheduler → Poll Supplier API → Transform → Kafka → Pipeline

Challenges:
- Rate limiting by suppliers
- API failures and retries
- Detecting what changed (no push notification from supplier)
```

### Pattern 4: File-based Ingestion (for bulk supplier feeds)
```
Supplier SFTP → File Watcher → Parse (XML/CSV) → Validate → Kafka/Direct Load

Common in OTA:
- Hotel content updates (descriptions, photos, amenities)
- Bulk rate uploads
- Inventory allocations
```

### Common Mistakes in Ingestion Design

| Mistake | Why It's Wrong | Correct Approach |
|---------|---------------|------------------|
| Writing directly to data lake from producers | No buffering, no replay, tightly coupled | Use Kafka as buffer |
| Single Kafka cluster for everything | Blast radius too large | Segment by use case |
| Not handling schema evolution | Pipeline breaks when upstream changes schema | Use Schema Registry (Avro/Protobuf) |
| Ignoring backpressure | Consumer falls behind, OOM, data loss | Monitor consumer lag, auto-scale |
| No dead letter queue (DLQ) | Bad messages poison the pipeline | Route unparseable messages to DLQ |
| Synchronous ingestion for analytics | Blocks the application on Kafka latency | Use 2-step async logging |

In [ ]:
# CDC Pipeline Implementation with Debezium + Kafka + Spark
# This pattern is very commonly asked in Agoda interviews

# --- Debezium CDC connector configuration (JSON) ---
debezium_config = {
    "name": "agoda-bookings-cdc",
    "config": {
        "connector.class": "io.debezium.connector.mysql.MySqlConnector",
        "database.hostname": "mysql-primary.agoda.internal",
        "database.port": "3306",
        "database.user": "cdc_reader",
        "database.server.id": "184054",
        "database.server.name": "bookings",
        "database.include.list": "booking_db",
        "table.include.list": "booking_db.bookings,booking_db.payments",
        
        # Schema evolution support
        "key.converter": "io.confluent.connect.avro.AvroConverter",
        "value.converter": "io.confluent.connect.avro.AvroConverter",
        "key.converter.schema.registry.url": "http://schema-registry:8081",
        "value.converter.schema.registry.url": "http://schema-registry:8081",
        
        # Exactly-once semantics
        "provide.transaction.metadata": "true",
        
        # Snapshot mode for initial load
        "snapshot.mode": "initial",
        
        # Handling deletes
        "tombstones.on.delete": "true",
        
        # Transforms for routing and filtering
        "transforms": "route,filter",
        "transforms.route.type": "org.apache.kafka.connect.transforms.RegexRouter",
        "transforms.route.regex": "([^.]+)\\.([^.]+)\\.([^.]+)",
        "transforms.route.replacement": "cdc.$3"
    }
}

print("=== Debezium CDC Configuration ===")
print(json.dumps(debezium_config, indent=2))
print()
print("Key Points for Interview:")
print("1. Reads MySQL binlog — zero impact on source database performance")
print("2. Avro + Schema Registry — handles schema evolution gracefully")
print("3. 'initial' snapshot — does full table scan first, then switches to binlog")
print("4. Transaction metadata — enables exactly-once in downstream processing")
print("5. Route transform — clean topic naming: cdc.bookings, cdc.payments")

# Part 5: Data Processing — Batch & Real-Time

## 5.1 Lambda vs Kappa Architecture

### Lambda Architecture (Most OTAs use this)
```
                    ┌─────────────────────┐
                    │   Batch Layer        │
     ┌─────────────▶│   (Spark Batch)      │──────┐
     │              │   Complete, accurate  │      │
     │              └─────────────────────┘      │
     │                                           ▼
 Data Source ─────────────────────────────▶ Serving Layer
     │                                           ▲
     │              ┌─────────────────────┐      │
     └─────────────▶│   Speed Layer        │──────┘
                    │   (Spark Streaming/  │
                    │    Flink)             │
                    │   Fast, approximate  │
                    └─────────────────────┘
```

### Kappa Architecture (Simpler, single path)
```
Data Source ──▶ Stream Processing ──▶ Serving Layer
                (everything is a stream)
```

### Which to choose? (Interview Answer)

| Criteria | Lambda | Kappa |
|----------|--------|-------|
| Complexity | Higher (two codepaths) | Lower (one codepath) |
| Accuracy | Batch corrects streaming errors | Must handle correctness in stream |
| Use case | Mixed latency requirements | Pure real-time |
| Reprocessing | Batch reprocesses easily | Replay from Kafka (retention limited) |
| **OTA recommendation** | **Preferred** — financial data needs batch correction | Good for pure clickstream analytics |

**Interview tip:** Say "We use Lambda for financial/booking pipelines (accuracy critical) and Kappa for clickstream/monitoring (speed critical)."

## 5.2 Apache Spark Deep Dive

In [ ]:
# ============================================================
# PySpark: Hotel Booking Analytics Pipeline
# This is the kind of code you should be able to discuss
# ============================================================

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, 
    DoubleType, TimestampType, LongType
)

# In a real environment:
# spark = SparkSession.builder \
#     .appName("AgodaBookingAnalytics") \
#     .config("spark.sql.adaptive.enabled", "true") \
#     .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
#     .config("spark.sql.shuffle.partitions", "200") \
#     .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
#     .getOrCreate()

# --- Schema for booking events ---
booking_schema = StructType([
    StructField("booking_id", LongType(), False),
    StructField("user_id", LongType(), False),
    StructField("property_id", LongType(), False),
    StructField("room_type_id", IntegerType(), False),
    StructField("check_in_date", StringType(), False),
    StructField("check_out_date", StringType(), False),
    StructField("booking_amount_usd", DoubleType(), False),
    StructField("currency", StringType(), False),
    StructField("status", StringType(), False),  # CREATED, CONFIRMED, CANCELLED
    StructField("channel", StringType(), False),  # WEB, APP, AGENT
    StructField("country", StringType(), False),
    StructField("city", StringType(), False),
    StructField("event_timestamp", TimestampType(), False),
    StructField("num_rooms", IntegerType(), False),
    StructField("num_guests", IntegerType(), False),
])

print("Schema defined. Key fields:")
for field in booking_schema.fields:
    print(f"  {field.name}: {field.dataType.simpleString()} {'(required)' if not field.nullable else ''}")

In [ ]:
# ============================================================
# Spark Optimization Techniques for TB-Scale Data
# CRITICAL INTERVIEW TOPIC
# ============================================================

spark_optimizations = """
╔══════════════════════════════════════════════════════════════════════╗
║             SPARK OPTIMIZATION CHEAT SHEET (Interview Ready)       ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                    ║
║  1. DATA SKEW (Most Common Problem at Scale)                       ║
║  ─────────────────────────────────────────────                     ║
║  Problem: Bangkok has 50K hotels, small city has 10.                ║
║           Partition by city → Bangkok partition is 5000x larger    ║
║                                                                    ║
║  Solutions:                                                        ║
║  a) Salting: Add random prefix to skewed key                       ║
║     df.withColumn("salted_city",                                   ║
║         concat(col("city"), lit("_"), (rand()*10).cast("int")))    ║
║     → Splits Bangkok into 10 partitions                           ║
║                                                                    ║
║  b) Broadcast join: If one side is small (<200MB default)          ║
║     df_big.join(broadcast(df_small), "key")                        ║
║                                                                    ║
║  c) AQE (Adaptive Query Execution) in Spark 3.x:                  ║
║     spark.sql.adaptive.enabled = true                              ║
║     spark.sql.adaptive.skewJoin.enabled = true                     ║
║     → Automatically splits skewed partitions at runtime            ║
║                                                                    ║
║  2. PARTITION STRATEGY                                             ║
║  ─────────────────────────                                         ║
║  • Write: Partition by date (year/month/day) for time-series       ║
║  • Read: Partition pruning — only read needed partitions           ║
║  • Avoid: Too many small partitions (<128MB) or too few large ones ║
║  • Target: 128MB-1GB per partition (optimal for Spark)             ║
║                                                                    ║
║  3. SHUFFLE OPTIMIZATION                                           ║
║  ──────────────────────                                            ║
║  • Reduce shuffles: Use map-side aggregation (reduceByKey > groupBy)║
║  • Tune: spark.sql.shuffle.partitions (default 200, often too few) ║
║  • AQE auto-coalesce: Merges small post-shuffle partitions         ║
║  • Bucketing: Pre-shuffle data at write time for repeated joins    ║
║                                                                    ║
║  4. MEMORY MANAGEMENT                                              ║
║  ────────────────────                                              ║
║  • spark.memory.fraction = 0.6 (execution + storage)              ║
║  • spark.memory.storageFraction = 0.5 (within memory.fraction)    ║
║  • OOM? → Increase partitions, reduce data per task               ║
║  • Cache wisely: Only cache if data is reused 2+ times            ║
║  • Use .persist(StorageLevel.DISK_ONLY) for large intermediate    ║
║                                                                    ║
║  5. FILE FORMAT OPTIMIZATION                                       ║
║  ──────────────────────────                                        ║
║  • Parquet: Columnar, best for analytical queries                  ║
║  • Delta Lake/Iceberg: ACID transactions, schema evolution, Z-order║
║  • Compaction: Merge small files into optimal-sized ones           ║
║  • Z-ordering: Co-locate related data for faster lookups           ║
║    delta_table.optimize().zOrderBy("property_id", "check_in_date") ║
║                                                                    ║
║  6. PREDICATE PUSHDOWN                                             ║
║  ────────────────────                                              ║
║  • Filter early: Push WHERE clauses down to storage layer          ║
║  • Parquet: Column pruning + row group filtering with statistics   ║
║  • Partition pruning: .where(col("date") == "2025-01-01")         ║
║    reads only that partition, not entire table                      ║
║                                                                    ║
╚══════════════════════════════════════════════════════════════════════╝
"""

print(spark_optimizations)

In [ ]:
# ============================================================
# PySpark: Complete Hotel Booking ETL Pipeline Example
# This demonstrates patterns you'd discuss in the interview
# ============================================================

# NOTE: This code is meant for study/discussion. To actually run it,
# you'd need a Spark cluster and the source data.

PIPELINE_CODE = '''
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

spark = SparkSession.builder \
    .appName("AgodaBookingETL") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.skewJoin.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .config("spark.sql.shuffle.partitions", "400") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()


# ---- STEP 1: Read raw booking events from data lake ----
raw_bookings = spark.read.format("delta").load("s3://agoda-datalake/raw/bookings/")

# ---- STEP 2: Data Quality Checks ----
quality_checks = raw_bookings.agg(
    F.count("*").alias("total_records"),
    F.sum(F.when(F.col("booking_id").isNull(), 1).otherwise(0)).alias("null_booking_ids"),
    F.sum(F.when(F.col("booking_amount_usd") < 0, 1).otherwise(0)).alias("negative_amounts"),
    F.sum(F.when(F.col("check_in_date") > F.col("check_out_date"), 1).otherwise(0)).alias("invalid_dates"),
    F.countDistinct("booking_id").alias("unique_bookings")
)

# Alert if duplicates detected (unique != total)
stats = quality_checks.collect()[0]
dup_rate = 1 - (stats["unique_bookings"] / stats["total_records"])
if dup_rate > 0.01:  # >1% duplicates = alert
    raise ValueError(f"Duplicate rate {dup_rate:.2%} exceeds threshold")


# ---- STEP 3: Deduplication ----
# Keep latest event per booking_id (exactly-once semantics)
window_spec = Window.partitionBy("booking_id").orderBy(F.col("event_timestamp").desc())

deduped = raw_bookings \
    .withColumn("row_num", F.row_number().over(window_spec)) \
    .filter(F.col("row_num") == 1) \
    .drop("row_num")


# ---- STEP 4: Enrichment ----
# Join with property dimension table (broadcast join for small table)
properties = spark.read.format("delta").load("s3://agoda-datalake/dim/properties/")

enriched = deduped.join(
    F.broadcast(properties.select(
        "property_id", "property_name", "star_rating", 
        "property_type", "region", "latitude", "longitude"
    )),
    on="property_id",
    how="left"
)


# ---- STEP 5: Derived columns ----
enriched = enriched \
    .withColumn("length_of_stay", 
        F.datediff(F.col("check_out_date"), F.col("check_in_date"))) \
    .withColumn("avg_nightly_rate", 
        F.col("booking_amount_usd") / F.col("length_of_stay")) \
    .withColumn("booking_date", 
        F.to_date("event_timestamp")) \
    .withColumn("lead_time_days",
        F.datediff(F.col("check_in_date"), F.col("booking_date"))) \
    .withColumn("is_weekend_stay",
        (F.dayofweek("check_in_date").isin([1, 7])) | 
        (F.dayofweek("check_out_date").isin([1, 7])))


# ---- STEP 6: Handle data skew ----
# Bangkok, Phuket, Tokyo dominate bookings — salt the city key
# for any city-level aggregations

city_agg = enriched \
    .withColumn("salt", (F.rand() * 10).cast("int")) \
    .withColumn("salted_city", F.concat(F.col("city"), F.lit("_"), F.col("salt"))) \
    .groupBy("salted_city") \
    .agg(
        F.count("*").alias("bookings"),
        F.avg("booking_amount_usd").alias("avg_amount"),
        F.avg("length_of_stay").alias("avg_los")
    ) \
    .withColumn("city", F.regexp_extract("salted_city", r"(.+)_\\d+", 1)) \
    .groupBy("city") \
    .agg(
        F.sum("bookings").alias("total_bookings"),
        F.avg("avg_amount").alias("avg_amount"),
        F.avg("avg_los").alias("avg_los")
    )


# ---- STEP 7: Write to Delta Lake with partitioning ----
enriched.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("booking_date", "country") \
    .option("overwriteSchema", "true") \
    .save("s3://agoda-datalake/curated/bookings/")


# ---- STEP 8: Optimize (compact small files, Z-order) ----
delta_table = DeltaTable.forPath(spark, "s3://agoda-datalake/curated/bookings/")
delta_table.optimize() \
    .where("booking_date >= current_date() - INTERVAL 7 DAYS") \
    .executeZOrderBy("property_id", "check_in_date")

# Vacuum old versions (keep 7 days for time travel)
delta_table.vacuum(168)  # 168 hours = 7 days
'''

print(PIPELINE_CODE)
print("\n" + "="*70)
print("KEY INTERVIEW TALKING POINTS:")
print("="*70)
print("""
1. AQE enabled for automatic optimization of shuffle partitions and skew handling
2. Data quality checks BEFORE processing (fail fast, don't propagate bad data)
3. Deduplication using window functions (handles at-least-once from Kafka)
4. Broadcast join for dimension tables (avoids shuffle)
5. Salt-based skew handling for city aggregations
6. Delta Lake for ACID transactions, schema evolution, time travel
7. Z-ordering by property_id + check_in_date (common query pattern)
8. Partition by date + country (matches query patterns)
9. Vacuum to manage storage costs (keep 7 days for debugging/rollback)
""")

In [ ]:
# ============================================================
# Spark Structured Streaming: Real-Time Pricing Pipeline
# ============================================================

STREAMING_CODE = '''
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import *

spark = SparkSession.builder \
    .appName("RealTimePricing") \
    .config("spark.sql.streaming.checkpointLocation", "s3://agoda-checkpoints/pricing/") \
    .getOrCreate()

price_schema = StructType([
    StructField("property_id", LongType()),
    StructField("room_type_id", IntegerType()),
    StructField("rate_plan_id", IntegerType()),
    StructField("date", StringType()),
    StructField("price_usd", DoubleType()),
    StructField("currency", StringType()),
    StructField("supplier_id", IntegerType()),
    StructField("updated_at", TimestampType()),
])

# Read from Kafka
raw_prices = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka-rt:9092") \
    .option("subscribe", "price-updates") \
    .option("startingOffsets", "latest") \
    .option("maxOffsetsPerTrigger", 1000000) \
    .option("failOnDataLoss", "false") \
    .load()

# Parse and transform
parsed = raw_prices \
    .select(F.from_json(F.col("value").cast("string"), price_schema).alias("data")) \
    .select("data.*") \
    .withWatermark("updated_at", "30 seconds")

# Deduplicate within watermark window (same property+room+date, keep latest)
deduped = parsed.dropDuplicates(["property_id", "room_type_id", "date"])

# Anomaly detection: flag suspicious price changes
# (prices dropping >80% or increasing >500% are likely errors)
# In production, compare against a rolling window or historical average

# Write to multiple sinks
# Sink 1: Delta Lake for analytics
query_delta = deduped.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "s3://checkpoints/pricing-delta/") \
    .partitionBy("date") \
    .trigger(processingTime="30 seconds") \
    .start("s3://agoda-datalake/streaming/prices/")

# Sink 2: Redis/ScyllaDB for online serving (via foreachBatch)
def write_to_feature_store(batch_df, batch_id):
    """Write latest prices to feature store for search ranking model."""
    latest_prices = batch_df \
        .groupBy("property_id", "room_type_id") \
        .agg(
            F.min("price_usd").alias("min_price"),
            F.avg("price_usd").alias("avg_price"),
            F.max("updated_at").alias("last_updated")
        )
    
    # Write to ScyllaDB/Redis (the feature store)
    latest_prices.write \
        .format("org.apache.spark.sql.cassandra") \
        .options(table="property_prices", keyspace="feature_store") \
        .mode("append") \
        .save()

query_feature_store = deduped.writeStream \
    .foreachBatch(write_to_feature_store) \
    .option("checkpointLocation", "s3://checkpoints/pricing-fs/") \
    .trigger(processingTime="10 seconds") \
    .start()

spark.streams.awaitAnyTermination()
'''

print(STREAMING_CODE)
print("\n" + "="*70)
print("STREAMING CONCEPTS TO KNOW:")
print("="*70)
print("""
1. WATERMARK: Tells Spark how late data can arrive.
   - "30 seconds" means events arriving >30s late are dropped
   - Without watermark: state grows unbounded → OOM
   - Too tight: drops legitimate late events
   - Too loose: high memory usage, delayed results

2. TRIGGER MODES:
   - processingTime="30s": micro-batch every 30 seconds
   - continuous (experimental): true record-by-record, ~1ms latency
   - once: process all available, then stop (for backfill)

3. OUTPUT MODES:
   - append: only new rows (most common for data lake writes)
   - update: changed rows only (good for feature store updates)
   - complete: full result table (only for aggregations)

4. EXACTLY-ONCE:
   - Checkpoint + WAL ensures exactly-once within Spark
   - Kafka offsets tracked in checkpoint
   - Delta Lake provides idempotent writes
   - External sinks (Redis) need idempotent write logic

5. BACKPRESSURE / maxOffsetsPerTrigger:
   - Limits records per trigger to prevent overwhelming the cluster
   - Critical during recovery: without it, Spark tries to process
     all accumulated data at once → OOM
""")

# Part 6: Data Modeling & Storage

## 6.1 Data Modeling for OTA Platform

### Medallion Architecture (Bronze → Silver → Gold)

```
┌─────────────────┐    ┌──────────────────┐    ┌──────────────────┐
│     BRONZE       │    │     SILVER        │    │      GOLD        │
│   (Raw Layer)    │───▶│  (Cleaned Layer)  │───▶│  (Business Layer)│
├─────────────────┤    ├──────────────────┤    ├──────────────────┤
│ • Raw events     │    │ • Deduplicated    │    │ • Aggregated     │
│ • As-is from     │    │ • Schema enforced │    │ • Business logic │
│   source         │    │ • Quality checked │    │   applied        │
│ • Append-only    │    │ • Enriched        │    │ • Ready for      │
│ • Full history   │    │ • Standardized    │    │   consumption    │
│                  │    │   types/formats   │    │ • Denormalized   │
│ Format: JSON/    │    │ Format: Delta/    │    │ Format: Delta/   │
│ Avro/raw         │    │ Parquet           │    │ Parquet          │
│ Retention: 90d+  │    │ Retention: 1-3yr  │    │ Retention: varies│
└─────────────────┘    └──────────────────┘    └──────────────────┘
```

### Why Medallion for Agoda?
- **Bronze**: Raw clickstream, booking events, price updates (debug/reprocessing)
- **Silver**: Clean booking facts, property dimensions, user profiles
- **Gold**: Daily booking aggregates, search ranking features, financial reports

## 6.2 Dimensional Modeling (Star Schema)

```
                    ┌──────────────────┐
                    │   dim_property    │
                    │ ──────────────── │
                    │ property_id (PK)  │
                    │ property_name     │
                    │ star_rating       │
                    │ property_type     │
                    │ city              │
                    │ country           │
                    │ latitude          │
                    │ longitude         │
                    └────────┬─────────┘
                             │
┌──────────────┐   ┌────────┴──────────┐   ┌──────────────┐
│  dim_date     │   │  fact_bookings     │   │  dim_user     │
│ ────────────  │   │ ──────────────── │   │ ────────────  │
│ date_key (PK) │───│ date_key (FK)     │───│ user_id (PK)  │
│ date          │   │ property_id (FK)  │   │ country       │
│ day_of_week   │   │ user_id (FK)      │   │ segment       │
│ month         │   │ channel_key (FK)  │   │ loyalty_tier  │
│ quarter       │   │ booking_id        │   │ signup_date   │
│ year          │   │ booking_amount    │   └──────────────┘
│ is_holiday    │   │ num_rooms         │
│ is_weekend    │   │ length_of_stay    │   ┌──────────────┐
└──────────────┘   │ lead_time_days    │   │ dim_channel   │
                    │ status            │   │ ────────────  │
                    │ cancellation_flag │───│ channel_key   │
                    └───────────────────┘   │ channel_name  │
                                            │ platform      │
                                            └──────────────┘
```

## 6.3 Storage Technology Selection

| Use Case | Technology | Why |
|----------|-----------|-----|
| OLTP (bookings, payments) | **MySQL/PostgreSQL** | ACID transactions, strong consistency |
| Data Lake (raw/curated) | **Delta Lake on S3/HDFS** | ACID, schema evolution, time travel |
| OLAP (analytics) | **Presto/Trino + Hive** | Fast SQL on petabyte-scale data |
| Feature Store (online) | **ScyllaDB + DragonflyDB** | Low-latency key-value at millions QPS |
| Search index | **Elasticsearch** | Full-text search, geo queries, faceted search |
| Caching | **Redis/DragonflyDB** | Sub-ms reads for hot data (prices, availability) |
| Time-series (metrics) | **InfluxDB/TimescaleDB** | Optimized for time-series writes and queries |
| Real-time OLAP | **Apache Druid/ClickHouse** | Sub-second analytics on streaming data |

### Common Mistake: Using One Storage for Everything
**Wrong**: "We'll store everything in Postgres."
**Right**: "We use polyglot persistence — different storage for different access patterns."

In [ ]:
# ============================================================
# Data Modeling: OTA Schema Examples
# ============================================================

# --- SQL Schema for Hotel Inventory (OLTP) ---
INVENTORY_SCHEMA_SQL = """
-- Hotel Inventory Management Schema (MySQL/PostgreSQL)
-- This is the OLTP schema used by the booking service

CREATE TABLE properties (
    property_id     BIGINT PRIMARY KEY,
    property_name   VARCHAR(255) NOT NULL,
    star_rating     TINYINT,
    property_type   ENUM('HOTEL', 'RESORT', 'APARTMENT', 'HOSTEL', 'VILLA'),
    city            VARCHAR(100) NOT NULL,
    country         VARCHAR(100) NOT NULL,
    latitude        DECIMAL(10, 7),
    longitude       DECIMAL(10, 7),
    timezone        VARCHAR(50),
    created_at      TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    updated_at      TIMESTAMP DEFAULT CURRENT_TIMESTAMP ON UPDATE CURRENT_TIMESTAMP,
    INDEX idx_city (city),
    INDEX idx_country_city (country, city)
);

CREATE TABLE room_types (
    room_type_id    INT PRIMARY KEY AUTO_INCREMENT,
    property_id     BIGINT NOT NULL,
    room_name       VARCHAR(255),
    max_occupancy   TINYINT,
    bed_type        VARCHAR(50),
    base_price_usd  DECIMAL(10, 2),
    FOREIGN KEY (property_id) REFERENCES properties(property_id)
);

-- Inventory table: one row per room_type per date
-- This is the hot table — millions of updates per day
CREATE TABLE inventory (
    property_id     BIGINT NOT NULL,
    room_type_id    INT NOT NULL,
    date            DATE NOT NULL,
    total_rooms     SMALLINT NOT NULL,
    booked_rooms    SMALLINT NOT NULL DEFAULT 0,
    blocked_rooms   SMALLINT NOT NULL DEFAULT 0,  -- maintenance, overbooking buffer
    price_usd       DECIMAL(10, 2) NOT NULL,
    min_stay        TINYINT DEFAULT 1,
    last_updated    TIMESTAMP DEFAULT CURRENT_TIMESTAMP ON UPDATE CURRENT_TIMESTAMP,
    version         INT DEFAULT 0,  -- for optimistic locking
    PRIMARY KEY (property_id, room_type_id, date),
    INDEX idx_date (date),
    INDEX idx_availability (date, property_id, booked_rooms)
);

-- Booking table with optimistic locking for concurrency
CREATE TABLE bookings (
    booking_id      BIGINT PRIMARY KEY AUTO_INCREMENT,
    user_id         BIGINT NOT NULL,
    property_id     BIGINT NOT NULL,
    room_type_id    INT NOT NULL,
    check_in        DATE NOT NULL,
    check_out       DATE NOT NULL,
    num_rooms       TINYINT NOT NULL DEFAULT 1,
    total_amount    DECIMAL(12, 2) NOT NULL,
    currency        CHAR(3) NOT NULL,
    status          ENUM('PENDING', 'CONFIRMED', 'CANCELLED', 'COMPLETED', 'NO_SHOW'),
    channel         ENUM('WEB', 'APP', 'AGENT', 'API'),
    idempotency_key VARCHAR(64) UNIQUE,  -- prevents duplicate bookings
    created_at      TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    updated_at      TIMESTAMP DEFAULT CURRENT_TIMESTAMP ON UPDATE CURRENT_TIMESTAMP,
    INDEX idx_user (user_id),
    INDEX idx_property_date (property_id, check_in, check_out),
    INDEX idx_status (status, created_at)
);
"""

print(INVENTORY_SCHEMA_SQL)

print("\n" + "="*70)
print("KEY DESIGN DECISIONS TO DISCUSS:")
print("="*70)
print("""
1. COMPOSITE PRIMARY KEY on inventory (property_id, room_type_id, date)
   → Natural key, efficient range scans by date

2. OPTIMISTIC LOCKING via 'version' column on inventory
   → UPDATE inventory SET booked_rooms = booked_rooms + 1, version = version + 1
     WHERE property_id = ? AND room_type_id = ? AND date = ? AND version = ?
   → If version doesn't match, someone else booked → retry

3. IDEMPOTENCY KEY on bookings
   → Client generates unique key; prevents double-booking on retry
   → Essential for payment safety

4. SEPARATE blocked_rooms column
   → Available = total_rooms - booked_rooms - blocked_rooms
   → Allows operations team to block rooms without changing bookings

5. INDEX STRATEGY:
   → idx_availability: Optimized for "find available rooms on date X"
   → idx_property_date: Optimized for "show all bookings for property"
""")

In [ ]:
# ============================================================
# Partitioning, Indexing & Data Retrieval Strategies
# ============================================================

partitioning_guide = """
╔══════════════════════════════════════════════════════════════════════╗
║          PARTITIONING & INDEXING STRATEGY GUIDE                    ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                    ║
║  DATA LAKE PARTITIONING                                            ║
║  ─────────────────────                                             ║
║                                                                    ║
║  Bookings: /year=2025/month=01/day=15/country=TH/                  ║
║    → Prune by date (most queries are date-bounded)                 ║
║    → Sub-partition by country (common filter)                      ║
║    → Z-order by property_id within partitions                      ║
║                                                                    ║
║  Clickstream: /year=2025/month=01/day=15/hour=14/                  ║
║    → Hour-level partitioning for high volume                       ║
║    → ~2.5B events per hour at Agoda's scale                        ║
║                                                                    ║
║  Prices: /date=2025-01-15/                                         ║
║    → Partition by stay_date (not update_date)                      ║
║    → Query pattern: "What's the price for Jan 15?"                 ║
║                                                                    ║
║  ANTI-PATTERNS                                                     ║
║  ──────────                                                        ║
║  ✗ Partitioning by high-cardinality column (user_id) → millions    ║
║    of tiny partitions, slow metadata operations                    ║
║  ✗ No partitioning → full table scans for every query              ║
║  ✗ Partition by booking_id → write-optimized, read-nightmare       ║
║  ✗ Too deep nesting (year/month/day/hour/country/city) → slow      ║
║    listing, too many small files                                   ║
║                                                                    ║
║  RULE OF THUMB: Each partition should be 128MB-1GB                 ║
║                                                                    ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                    ║
║  DATABASE SHARDING (for OLTP at Scale)                             ║
║  ────────────────────────────────────                              ║
║                                                                    ║
║  Inventory table: Shard by property_id                             ║
║    → All dates for a property on same shard                        ║
║    → Booking transaction = single-shard transaction (fast)         ║
║                                                                    ║
║  Bookings table: Shard by booking_id                               ║
║    → Even distribution (auto-increment)                            ║
║    → Trade-off: cross-shard query for "all bookings for property"  ║
║                                                                    ║
║  Alternative: Shard by property_id for both                        ║
║    → Booking + inventory update in same shard = local transaction  ║
║    → Trade-off: uneven shard sizes (popular properties)            ║
║                                                                    ║
║  INDEXING STRATEGIES                                               ║
║  ───────────────────                                               ║
║                                                                    ║
║  Elasticsearch (for search):                                       ║
║    → Denormalized documents (property + rooms + prices)            ║
║    → Geo-point field for location-based search                     ║
║    → Keyword fields for exact match (city, country)                ║
║    → Nested objects for room types                                 ║
║    → Alias-based reindexing for zero-downtime updates              ║
║                                                                    ║
║  Redis (for availability):                                         ║
║    → Key: property:{id}:room:{type}:avail                         ║
║    → Value: Bitmap per month (bit per day: 1=available)            ║
║    → Or Sorted Set: score=date, member=available_count             ║
║    → TTL: 90 days (no need to cache far-future dates)              ║
║                                                                    ║
╚══════════════════════════════════════════════════════════════════════╝
"""

print(partitioning_guide)

## 6.4 Consistency Patterns (Critical for Booking Systems)

### The Double-Booking Problem

```
User A: Check availability → 1 room left → Book → Success
User B: Check availability → 1 room left → Book → ??? (Race condition!)
```

### Solutions (Ordered by Preference for OTA)

**1. Optimistic Locking (Recommended for Agoda-scale)**
```sql
-- Read current version
SELECT available_rooms, version FROM inventory 
WHERE property_id = 123 AND room_type_id = 1 AND date = '2025-01-15';

-- Update with version check (CAS - Compare And Swap)
UPDATE inventory 
SET booked_rooms = booked_rooms + 1, version = version + 1
WHERE property_id = 123 AND room_type_id = 1 AND date = '2025-01-15'
  AND version = 42 AND (total_rooms - booked_rooms - blocked_rooms) >= 1;

-- If affected_rows = 0 → someone else booked → retry with backoff
```

**2. Pessimistic Locking (For low-volume, high-value bookings)**
```sql
BEGIN;
SELECT * FROM inventory WHERE property_id = 123 AND date = '2025-01-15' FOR UPDATE;
-- Lock held until transaction completes
-- Other transactions block here
UPDATE inventory SET booked_rooms = booked_rooms + 1 ...;
INSERT INTO bookings ...;
COMMIT;
```

**3. Reservation Pattern (Agoda likely uses this)**
```
Step 1: RESERVE (hold for 10 minutes, decrement available)
Step 2: User fills payment details
Step 3: CONFIRM (finalize booking) or EXPIRE (release inventory)

Advantages:
- User has time to complete payment without losing room
- Expired reservations auto-release inventory
- Reduces contention on the inventory table
```

### CAP Theorem for OTA

| System | Choose | Why |
|--------|--------|-----|
| Booking/Payment | **CP** (Consistency + Partition tolerance) | Cannot double-book or lose money |
| Search/Listing | **AP** (Availability + Partition tolerance) | Slightly stale results OK |
| Price Display | **AP** | Eventually consistent pricing acceptable |
| Inventory Count | **CP** for booking, **AP** for display | Show "limited availability" even if count is approximate |

# Part 7: Feature Store & ML Pipelines

## 7.1 Agoda's Feature Store Architecture

This is a major topic — Agoda has extensively blogged about their feature store.

```
┌──────────────────────────────────────────────────────────────────┐
│                    AGODA FEATURE STORE                           │
├──────────────────────────────────────────────────────────────────┤
│                                                                  │
│  OFFLINE PATH (Training)                                         │
│  ┌─────────┐    ┌───────────┐    ┌──────────┐    ┌──────────┐  │
│  │ Data    │───▶│ Spark     │───▶│ Feature  │───▶│ Model    │  │
│  │ Lake    │    │ Jobs      │    │ Tables   │    │ Training │  │
│  │ (HDFS)  │    │ (Batch)   │    │ (Hive)   │    │          │  │
│  └─────────┘    └───────────┘    └──────────┘    └──────────┘  │
│                                                                  │
│  ONLINE PATH (Inference) — 3.5M entities/sec                    │
│  ┌─────────┐    ┌───────────┐    ┌──────────┐    ┌──────────┐  │
│  │ Kafka   │───▶│ Flink/    │───▶│ ScyllaDB │───▶│ Feature  │  │
│  │ Stream  │    │ Spark SS  │    │ (source  │    │ Serving  │  │
│  │         │    │           │    │  of truth)│    │ (Rust)   │  │
│  └─────────┘    └───────────┘    └──────────┘    └──────────┘  │
│                                       │              │          │
│                                       ▼              ▼          │
│                                  ┌──────────┐  ┌──────────┐    │
│                                  │Dragonfly │  │ Client   │    │
│                                  │DB Cache  │  │ SDK Cache│    │
│                                  │(central) │  │ (local)  │    │
│                                  │ P99 <3ms │  │          │    │
│                                  └──────────┘  └──────────┘    │
│                                                                  │
│  PERFORMANCE:                                                    │
│  • 3.5M entities/sec total throughput                            │
│  • ~50% served from client SDK cache                             │
│  • 1.7M entities/sec reaching application servers                │
│  • P99 latency < 10ms SLA                                       │
│  • DragonflyDB cache P99 < 3ms                                   │
│  • 50x scaling in 2 years (Jan 2023 → Feb 2025)                 │
│  • Rust-based application layer on Kubernetes                    │
│  • ScyllaDB as source of truth (CQL-compatible)                  │
│                                                                  │
└──────────────────────────────────────────────────────────────────┘
```

## 7.2 Feature Types in Hotel Search Ranking

| Feature Type | Examples | Update Frequency | Storage |
|-------------|----------|-----------------|----------|
| **Static** | Hotel location, star rating, amenities | Days/weeks | Data Lake → Feature Store |
| **Slow-changing** | Review score, average rating, popularity | Hours | Batch Spark → Feature Store |
| **Real-time** | Current price, availability, recent clicks | Seconds | Streaming → Feature Store |
| **User-specific** | Past bookings, search history, preferences | Minutes | Streaming → Feature Store |
| **Context** | Time of day, day of week, seasonality | Request-time | Computed at inference |

In [ ]:
# ============================================================
# Feature Engineering Pipeline for Hotel Search Ranking
# This is a common Agoda interview question
# ============================================================

FEATURE_PIPELINE_CODE = '''
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("HotelFeatureEngineering").getOrCreate()

# ---- Load source data ----
bookings = spark.read.format("delta").load("s3://datalake/silver/bookings/")
reviews = spark.read.format("delta").load("s3://datalake/silver/reviews/")
clicks = spark.read.format("delta").load("s3://datalake/silver/clickstream/")
properties = spark.read.format("delta").load("s3://datalake/silver/properties/")


# ---- Feature Group 1: Property Popularity Features ----
# Rolling 30-day booking counts and conversion rates
window_30d = Window.partitionBy("property_id").orderBy("date").rangeBetween(-30*86400, 0)

property_popularity = bookings \
    .groupBy("property_id", F.to_date("event_timestamp").alias("date")) \
    .agg(
        F.count("*").alias("daily_bookings"),
        F.sum("booking_amount_usd").alias("daily_revenue"),
        F.countDistinct("user_id").alias("daily_unique_bookers")
    ) \
    .withColumn("bookings_30d", F.sum("daily_bookings").over(window_30d)) \
    .withColumn("revenue_30d", F.sum("daily_revenue").over(window_30d)) \
    .withColumn("unique_bookers_30d", F.sum("daily_unique_bookers").over(window_30d))


# ---- Feature Group 2: Review Features ----
review_features = reviews \
    .groupBy("property_id") \
    .agg(
        F.count("*").alias("total_reviews"),
        F.avg("overall_score").alias("avg_review_score"),
        F.avg("cleanliness_score").alias("avg_cleanliness"),
        F.avg("location_score").alias("avg_location_score"),
        F.avg("value_score").alias("avg_value_score"),
        F.stddev("overall_score").alias("review_score_stddev"),
        # Recency-weighted score (recent reviews matter more)
        F.sum(
            F.col("overall_score") * 
            F.exp(-0.01 * F.datediff(F.current_date(), F.col("review_date")))
        ).alias("recency_weighted_score_sum"),
        F.sum(
            F.exp(-0.01 * F.datediff(F.current_date(), F.col("review_date")))
        ).alias("recency_weight_sum")
    ) \
    .withColumn("recency_weighted_score", 
        F.col("recency_weighted_score_sum") / F.col("recency_weight_sum"))


# ---- Feature Group 3: Click-Through & Conversion Features ----
ctr_features = clicks \
    .filter(F.col("event_type").isin(["impression", "click", "booking"])) \
    .groupBy("property_id") \
    .agg(
        F.sum(F.when(F.col("event_type") == "impression", 1).otherwise(0)).alias("impressions"),
        F.sum(F.when(F.col("event_type") == "click", 1).otherwise(0)).alias("clicks"),
        F.sum(F.when(F.col("event_type") == "booking", 1).otherwise(0)).alias("conversions")
    ) \
    .withColumn("ctr", F.col("clicks") / F.col("impressions")) \
    .withColumn("cvr", F.col("conversions") / F.col("clicks")) \
    .withColumn("booking_rate", F.col("conversions") / F.col("impressions"))


# ---- Feature Group 4: Price Competitiveness ----
prices = spark.read.format("delta").load("s3://datalake/silver/prices/")

price_features = prices \
    .groupBy("property_id") \
    .agg(
        F.avg("price_usd").alias("avg_price"),
        F.min("price_usd").alias("min_price"),
        F.stddev("price_usd").alias("price_volatility")
    )

# City-level price percentile for competitiveness
city_window = Window.partitionBy("city")
price_with_city = prices.join(F.broadcast(properties.select("property_id", "city")), "property_id")
price_competitiveness = price_with_city \
    .withColumn("city_avg_price", F.avg("price_usd").over(city_window)) \
    .withColumn("price_vs_city_avg", F.col("price_usd") / F.col("city_avg_price")) \
    .groupBy("property_id") \
    .agg(F.avg("price_vs_city_avg").alias("avg_price_competitiveness"))


# ---- Join all features ----
feature_table = properties \
    .join(property_popularity, "property_id", "left") \
    .join(review_features, "property_id", "left") \
    .join(ctr_features, "property_id", "left") \
    .join(price_features, "property_id", "left") \
    .join(price_competitiveness, "property_id", "left")

# Fill nulls with sensible defaults (cold start properties)
feature_table = feature_table.na.fill({
    "bookings_30d": 0,
    "avg_review_score": 3.0,   # neutral default
    "total_reviews": 0,
    "ctr": 0.01,               # low default CTR
    "cvr": 0.001,              # low default CVR
})

# Write to offline feature store (for model training)
feature_table.write.format("delta").mode("overwrite") \
    .save("s3://datalake/gold/search_ranking_features/")

# Write to online feature store (ScyllaDB for real-time serving)
feature_table.select(
    "property_id", "bookings_30d", "avg_review_score", 
    "ctr", "cvr", "avg_price_competitiveness"
).write \
    .format("org.apache.spark.sql.cassandra") \
    .options(table="property_features", keyspace="feature_store") \
    .mode("append") \
    .save()
'''

print(FEATURE_PIPELINE_CODE)
print("\n" + "="*70)
print("FEATURE ENGINEERING INTERVIEW TIPS:")
print("="*70)
print("""
1. Always discuss OFFLINE vs ONLINE feature consistency
   → Training features must match serving features (training-serving skew)

2. Cold start handling is a MUST-discuss topic
   → New properties have no bookings/reviews → use sensible defaults
   → Explore-exploit: boost new properties to collect signal

3. Feature freshness matters
   → Static features (star rating): daily refresh is fine
   → Real-time features (price, availability): seconds matter
   → Discuss how each feature type gets updated

4. Point-in-time correctness
   → For training: features must be computed as-of booking time
   → Wrong: using current review score to predict past bookings
   → Right: use review score as it was at the time of the booking

5. Feature monitoring
   → Track feature distributions over time
   → Alert on drift (indicates data quality issues or concept shift)
""")

# Part 8: Performance, Scalability & Optimization

## 8.1 Scaling Patterns for 1M+ QPS

### Horizontal vs Vertical Scaling

| Pattern | When to Use | Example at Agoda |
|---------|------------|------------------|
| **Horizontal scaling** | Stateless services, partitioned data | Add more Kafka brokers, Spark executors |
| **Vertical scaling** | Single bottleneck, before horizontal | Bigger DragonflyDB instance (64-core) |
| **Caching** | Read-heavy, data can be stale | Feature Store cache (DragonflyDB P99 <3ms) |
| **Sharding** | Data too large for single node | Inventory by property_id, ScyllaDB vnodes |
| **Read replicas** | Read-heavy OLTP | MySQL read replicas for search queries |
| **CQRS** | Different read/write patterns | Write to MySQL, read from Elasticsearch |
| **Async processing** | Tasks don't need immediate result | Booking confirmation email via Kafka |

### Caching Strategy (Multi-Layer)

```
Request → [CDN Cache] → [App-level Cache] → [Centralized Cache] → [Database]
          (static assets)  (in-process,       (Redis/Dragonfly,    (ScyllaDB/
           TTL: hours       local memory)       TTL: seconds-min)   MySQL)
                            TTL: seconds)

Agoda's Feature Store uses 3 layers:
1. Client SDK cache (in-process) — serves ~50% of requests
2. DragonflyDB (centralized cache) — P99 <3ms
3. ScyllaDB (source of truth) — P99 <10ms
```

### Cache Invalidation Strategies

| Strategy | How It Works | Pros | Cons |
|----------|-------------|------|------|
| **TTL** | Cache entry expires after N seconds | Simple, self-healing | Stale data within TTL window |
| **Write-through** | Update cache on every write | Always fresh | Higher write latency |
| **Write-behind** | Async cache update after write | Low write latency | Temporarily stale |
| **CDC-based** | Database change triggers cache update | Near-real-time freshness | More complex |
| **Event-driven** | Kafka event triggers cache invalidation | Decoupled, scalable | Eventual consistency |

**For OTA pricing**: Event-driven (price update → Kafka → invalidate cache) + short TTL as safety net.

## 8.2 Identifying & Resolving Bottlenecks

In [ ]:
# ============================================================
# Bottleneck Identification & Resolution Guide
# ============================================================

bottlenecks = {
    "Kafka Consumer Lag": {
        "symptom": "Consumer group offset falling behind producer offset",
        "cause": [
            "Too few consumers in group",
            "Slow processing per message",
            "Uneven partition assignment",
            "GC pauses in consumer"
        ],
        "resolution": [
            "Add consumers (max = num partitions)",
            "Increase partition count + consumers together",
            "Optimize processing (batch DB writes, async I/O)",
            "Use incremental rebalancing (cooperative-sticky assignor)",
            "Tune fetch.min.bytes and max.poll.records"
        ],
        "metric_to_monitor": "consumer_lag_records, consumer_lag_time_ms"
    },
    
    "Spark Job Slow / OOM": {
        "symptom": "Tasks stuck, executor OOM, job takes hours instead of minutes",
        "cause": [
            "Data skew (one partition 100x larger than others)",
            "Too few partitions (200 default, need 2000+)",
            "Cartesian join (missing join key → full cross product)",
            "Collecting large DataFrame to driver",
            "Unoptimized UDFs (Python UDFs are 10-100x slower than built-in)"
        ],
        "resolution": [
            "Enable AQE: spark.sql.adaptive.enabled=true",
            "Salt skewed keys (add random prefix, aggregate twice)",
            "Increase shuffle.partitions proportional to data size",
            "Use broadcast join for small tables (<200MB)",
            "Replace Python UDFs with Spark SQL functions or Pandas UDFs",
            "Use .explain() and Spark UI to find the bottleneck stage"
        ],
        "metric_to_monitor": "stage_duration, task_duration_distribution, shuffle_bytes, gc_time"
    },
    
    "Database Hot Partition": {
        "symptom": "One DB shard at 100% CPU while others idle",
        "cause": [
            "Popular property (Bangkok Hilton) gets 1000x more queries",
            "Sequential key generation causing write hotspot",
            "Time-based partitioning with most queries hitting 'today'"
        ],
        "resolution": [
            "Add read replicas for hot shards",
            "Cache hot data in Redis/Dragonfly",
            "Use consistent hashing with virtual nodes for better distribution",
            "Split hot shard: sub-shard popular properties",
            "For ScyllaDB: enable tablets for automatic load balancing"
        ],
        "metric_to_monitor": "per_shard_qps, per_shard_latency_p99, cpu_per_node"
    },
    
    "Small File Problem (Data Lake)": {
        "symptom": "Queries slow despite small data volume; metadata operations slow",
        "cause": [
            "Streaming writes creating thousands of tiny files",
            "Over-partitioning (too many partition columns)",
            "Frequent small batch writes"
        ],
        "resolution": [
            "Delta Lake OPTIMIZE (compaction): merge small files",
            "Auto-compaction: delta.autoOptimize.optimizeWrite=true",
            "Reduce partition granularity (daily instead of hourly if volume is low)",
            "Use streaming trigger interval of 1-5 min (not seconds)",
            "Target: 128MB-1GB per file"
        ],
        "metric_to_monitor": "avg_file_size, num_files_per_partition, list_operation_time"
    },
    
    "Feature Serving Latency Spike": {
        "symptom": "P99 latency >10ms SLA, intermittent timeouts",
        "cause": [
            "Cache miss storm (after deployment or cache restart)",
            "GC pauses (if using JVM-based cache)",
            "Network partition between cache and DB",
            "Bursty traffic pattern (flash sale)"
        ],
        "resolution": [
            "Cache warming: pre-populate cache before deployment",
            "Agoda chose DragonflyDB (no GC — C++ based) over Redis",
            "Circuit breaker: fall back to stale cache on DB timeout",
            "Rate limiting: protect feature store from traffic spikes",
            "Multi-tier caching: client SDK cache absorbs 50% of load"
        ],
        "metric_to_monitor": "p50/p95/p99 latency, cache_hit_rate, error_rate"
    }
}

for name, details in bottlenecks.items():
    print(f"\n{'='*60}")
    print(f"BOTTLENECK: {name}")
    print(f"{'='*60}")
    print(f"Symptom: {details['symptom']}")
    print(f"\nCauses:")
    for c in details['cause']:
        print(f"  • {c}")
    print(f"\nResolutions:")
    for r in details['resolution']:
        print(f"  ✓ {r}")
    print(f"\nMonitor: {details['metric_to_monitor']}")

# Part 9: Operational Excellence

## 9.1 Monitoring & Observability

### The Four Pillars of Observability

```
┌──────────────┐  ┌──────────────┐  ┌──────────────┐  ┌──────────────┐
│   METRICS    │  │    LOGS      │  │   TRACES     │  │   ALERTS     │
├──────────────┤  ├──────────────┤  ├──────────────┤  ├──────────────┤
│ • Throughput  │  │ • Structured │  │ • Distributed│  │ • Tiered     │
│ • Latency    │  │   JSON logs  │  │   tracing    │  │   (P1-P4)   │
│   (p50/p99)  │  │ • Log levels │  │ • Span IDs   │  │ • Escalation │
│ • Error rate │  │ • Correlation│  │ • End-to-end │  │   policies   │
│ • Saturation │  │   IDs        │  │   latency    │  │ • Runbooks   │
│              │  │              │  │   breakdown  │  │ • Auto-      │
│ Tool:        │  │ Tool:        │  │ Tool:        │  │   remediation│
│ Prometheus/  │  │ ELK/Loki    │  │ Jaeger/      │  │ Tool:        │
│ Grafana      │  │              │  │ Zipkin       │  │ PagerDuty    │
└──────────────┘  └──────────────┘  └──────────────┘  └──────────────┘
```

### Key Metrics for Data Pipelines

| Category | Metric | Alert Threshold (Example) |
|----------|--------|--------------------------|
| **Freshness** | Time since last successful run | >2x normal interval |
| **Volume** | Records processed per run | <80% or >150% of expected |
| **Quality** | Null rate, duplicate rate, schema violations | >1% null rate on required fields |
| **Latency** | End-to-end pipeline latency | >SLA threshold |
| **Consumer Lag** | Kafka consumer offset lag | >100K messages behind |
| **Error Rate** | Failed records / total records | >0.1% |
| **Resource** | CPU, memory, disk utilization | >80% sustained |
| **Completeness** | Expected vs actual partition count | Missing partitions |

### Agoda's Three-Tier Alerting System

```
Tier 1 (CRITICAL): Data pipeline completely down, data loss risk
  → Immediate page, auto-escalation after 15 min
  → Example: Kafka cluster unavailable, Spark job failing repeatedly

Tier 2 (WARNING): Degraded performance, SLA at risk
  → Slack notification, business hours response
  → Example: Consumer lag >10 min, data quality anomaly detected

Tier 3 (INFO): Non-urgent issues, optimization opportunities
  → Dashboard update, weekly review
  → Example: Storage approaching capacity, query performance degradation
```

## 9.2 Data Quality Framework

In [ ]:
# ============================================================
# Data Quality Monitoring Framework
# This is a critical topic for Agoda interviews
# ============================================================

from dataclasses import dataclass, field
from enum import Enum
from typing import Callable, Any
from datetime import datetime


class Severity(Enum):
    CRITICAL = "CRITICAL"  # Block pipeline, page on-call
    WARNING = "WARNING"    # Alert, investigate within business hours
    INFO = "INFO"          # Log for weekly review


class CheckType(Enum):
    COMPLETENESS = "completeness"   # No missing required fields
    FRESHNESS = "freshness"         # Data is recent enough
    VOLUME = "volume"               # Expected number of records
    UNIQUENESS = "uniqueness"       # No unexpected duplicates
    VALIDITY = "validity"           # Values in expected range/format
    CONSISTENCY = "consistency"     # Cross-table referential integrity
    DISTRIBUTION = "distribution"   # Statistical distribution checks


@dataclass
class DataQualityCheck:
    name: str
    check_type: CheckType
    severity: Severity
    table: str
    column: str = ""
    threshold: float = 0.0
    description: str = ""
    sql: str = ""


# Define quality checks for the booking pipeline
BOOKING_QUALITY_CHECKS = [
    DataQualityCheck(
        name="booking_id_uniqueness",
        check_type=CheckType.UNIQUENESS,
        severity=Severity.CRITICAL,
        table="silver.bookings",
        column="booking_id",
        threshold=0.0,
        description="Booking IDs must be unique — duplicates indicate pipeline bug",
        sql="""SELECT COUNT(*) - COUNT(DISTINCT booking_id) as dup_count 
               FROM silver.bookings WHERE booking_date = '{date}'"""
    ),
    DataQualityCheck(
        name="daily_volume_check",
        check_type=CheckType.VOLUME,
        severity=Severity.CRITICAL,
        table="silver.bookings",
        threshold=0.5,  # alert if <50% of expected volume
        description="Daily bookings should not drop more than 50% — indicates data loss",
        sql="""SELECT COUNT(*) as today_count,
                      (SELECT AVG(cnt) FROM (
                          SELECT COUNT(*) as cnt FROM silver.bookings 
                          WHERE booking_date BETWEEN '{date}' - INTERVAL 7 DAY AND '{date}' - INTERVAL 1 DAY
                          GROUP BY booking_date
                      ) t) as avg_7d
               FROM silver.bookings WHERE booking_date = '{date}'"""
    ),
    DataQualityCheck(
        name="amount_validity",
        check_type=CheckType.VALIDITY,
        severity=Severity.WARNING,
        table="silver.bookings",
        column="booking_amount_usd",
        threshold=0.001,  # <0.1% invalid
        description="Booking amounts should be positive and < $100,000",
        sql="""SELECT SUM(CASE WHEN booking_amount_usd <= 0 OR booking_amount_usd > 100000 THEN 1 ELSE 0 END) * 1.0 
                      / COUNT(*) as invalid_rate
               FROM silver.bookings WHERE booking_date = '{date}'"""
    ),
    DataQualityCheck(
        name="date_consistency",
        check_type=CheckType.VALIDITY,
        severity=Severity.WARNING,
        table="silver.bookings",
        description="Check-out date must be after check-in date",
        sql="""SELECT SUM(CASE WHEN check_out_date <= check_in_date THEN 1 ELSE 0 END) as invalid_dates
               FROM silver.bookings WHERE booking_date = '{date}'"""
    ),
    DataQualityCheck(
        name="property_referential_integrity",
        check_type=CheckType.CONSISTENCY,
        severity=Severity.WARNING,
        table="silver.bookings",
        column="property_id",
        description="Every booking should reference a valid property",
        sql="""SELECT COUNT(*) as orphan_bookings
               FROM silver.bookings b
               LEFT JOIN silver.properties p ON b.property_id = p.property_id
               WHERE b.booking_date = '{date}' AND p.property_id IS NULL"""
    ),
    DataQualityCheck(
        name="freshness_check",
        check_type=CheckType.FRESHNESS,
        severity=Severity.CRITICAL,
        table="silver.bookings",
        threshold=7200,  # 2 hours max staleness
        description="Data should not be more than 2 hours old",
        sql="""SELECT TIMESTAMPDIFF(SECOND, MAX(event_timestamp), NOW()) as staleness_seconds
               FROM silver.bookings"""
    ),
    DataQualityCheck(
        name="price_distribution_anomaly",
        check_type=CheckType.DISTRIBUTION,
        severity=Severity.WARNING,
        table="silver.bookings",
        column="booking_amount_usd",
        description="Average booking amount should not deviate >3 std from 30-day rolling avg",
        sql="""WITH daily_avg AS (
                   SELECT AVG(booking_amount_usd) as avg_amount
                   FROM silver.bookings WHERE booking_date = '{date}'
               ),
               historical AS (
                   SELECT AVG(booking_amount_usd) as hist_avg,
                          STDDEV(booking_amount_usd) as hist_std
                   FROM silver.bookings 
                   WHERE booking_date BETWEEN '{date}' - INTERVAL 30 DAY AND '{date}' - INTERVAL 1 DAY
               )
               SELECT ABS(d.avg_amount - h.hist_avg) / h.hist_std as z_score
               FROM daily_avg d, historical h"""
    )
]


print("=== BOOKING PIPELINE DATA QUALITY CHECKS ===")
print(f"Total checks: {len(BOOKING_QUALITY_CHECKS)}\n")

for check in BOOKING_QUALITY_CHECKS:
    severity_marker = {"CRITICAL": "!!!", "WARNING": "!!", "INFO": "!"}[check.severity.value]
    print(f"[{severity_marker} {check.severity.value}] {check.name}")
    print(f"    Type: {check.check_type.value}")
    print(f"    {check.description}")
    print()

## 9.3 Failure Recovery & Disaster Recovery

### Failure Scenarios and Recovery Strategies

| Failure | Impact | Recovery Strategy |
|---------|--------|-------------------|
| **Spark job fails midway** | Partial data in output | Idempotent writes (Delta Lake overwrite), checkpoint restart |
| **Kafka broker down** | Reduced throughput | Replication factor=3, ISR (In-Sync Replicas) ensures no data loss |
| **Schema change upstream** | Pipeline crashes on parse | Schema Registry with backward compatibility, DLQ for unparseable |
| **Data lake corruption** | Bad data served to consumers | Delta Lake time travel (rollback to previous version), audit log |
| **Feature store outage** | ML models can't get features | Multi-tier cache (client SDK cache serves stale features) |
| **Full data center outage** | Complete service loss | Active-active data centers, Kafka MirrorMaker for cross-DC replication |

### Backfill Strategy (Very Common Interview Question)

```
Scenario: You discover a bug in your Spark ETL that has been producing
          incorrect booking amounts for the last 3 months.

Step 1: Fix the bug in the ETL code

Step 2: Estimate backfill scope
  - 3 months × ~100K bookings/day = ~9M records
  - Source data in Bronze layer (raw, immutable) = still correct

Step 3: Run backfill job
  - Read from Bronze (raw) layer — this is why we keep raw data!
  - Process with fixed ETL logic
  - Write to Silver layer using Delta Lake MERGE (upsert)
  
Step 4: Validate
  - Compare row counts: backfill output vs original
  - Spot-check: sample records and verify manually
  - Run data quality checks on backfilled data
  
Step 5: Propagate to downstream (Gold layer, Feature Store)
  - Trigger Gold layer rebuilds for affected date range
  - Update Feature Store with corrected features
  - Invalidate any cached data

Step 6: Post-mortem
  - Add data quality check that would have caught this bug
  - Add unit test for the fixed transformation
```

### Exactly-Once Processing (End-to-End)

```
Kafka (at-least-once) → Spark (exactly-once via checkpoint) → Delta Lake (idempotent writes)

Key mechanisms:
1. Kafka: Enable idempotent producer (acks=all, enable.idempotence=true)
2. Spark: Checkpoint + WAL track processed offsets
3. Delta Lake: MERGE with matching condition = idempotent upsert
4. External sinks: Idempotency key (booking_id + timestamp) in the target
```

# Part 10: Full System Design Walkthroughs

## Design 1: Real-Time Hotel Pricing Pipeline

**Problem**: Design a system that processes rate updates from 2M+ hotel properties and serves fresh prices to search results within 5 seconds.

### Step 1: Clarifying Questions & Requirements

```
Functional Requirements:
- Ingest price updates from multiple suppliers (GDS, direct, OTA partners)
- Process and normalize prices (currency conversion, tax calculation)
- Serve current prices for search results with <5s freshness
- Support historical price analytics (trend analysis, competitive monitoring)

Non-Functional Requirements:
- 2M properties × 10 room types × 365 days = ~7.3B price cells
- ~50M price updates per hour during peak
- Freshness SLA: 5 seconds from update to searchable
- Availability: 99.99% (search cannot go down)
- Latency: <10ms for price lookup (single property)
```

### Step 2: Back-of-Envelope

```
Peak QPS: 50M updates/hour ÷ 3600 = ~14K updates/sec
Read QPS: 10x write = ~140K reads/sec (search queries)
Avg event size: ~500 bytes
Throughput: 14K × 500B = 7 MB/sec (very manageable for Kafka)
Storage: 7.3B cells × 200B = ~1.5TB (fits in memory across cluster)
```

### Step 3: High-Level Architecture

```
┌────────────┐  ┌────────────┐  ┌────────────┐
│ Supplier A │  │ Supplier B │  │ Direct API │
│ (GDS/XML)  │  │ (REST/JSON)│  │ (Webhook)  │
└─────┬──────┘  └─────┬──────┘  └─────┬──────┘
      │               │               │
      ▼               ▼               ▼
┌──────────────────────────────────────────┐
│         Ingestion Gateway                │
│  • Rate limiting per supplier            │
│  • Schema validation                     │
│  • Normalize to canonical format          │
│  • Deduplicate (idempotency key)         │
└─────────────────┬────────────────────────┘
                  │
                  ▼
┌──────────────────────────────────────────┐
│         Kafka: price-updates              │
│         (partition by property_id)        │
│         ~200 partitions                   │
└────┬──────────────────────────┬──────────┘
     │                          │
     ▼                          ▼
┌──────────────┐       ┌──────────────────┐
│ Spark        │       │ Flink/Spark SS   │
│ Streaming    │       │ (Real-time path) │
│ (Analytics)  │       │ • Currency conv  │
│ • Store to   │       │ • Tax calc       │
│   Delta Lake │       │ • Anomaly detect │
│ • Historical │       │ • Write to cache │
└──────┬───────┘       └────────┬─────────┘
       │                        │
       ▼                        ▼
┌──────────────┐       ┌──────────────────┐
│ Delta Lake   │       │ Redis Cluster    │
│ (analytics   │       │ (serving cache)  │
│  & backfill) │       │ TTL: 1 hour      │
│              │       │ Key: prop:room:dt │
└──────────────┘       └────────┬─────────┘
                                │
                                ▼
                       ┌──────────────────┐
                       │ Price Serving API │
                       │ • Read from Redis │
                       │ • Fallback to DB  │
                       │ • <10ms P99       │
                       └──────────────────┘
```

### Step 4: Deep Dive — Key Design Decisions

**Why partition Kafka by property_id?**
- All price updates for a property go to the same partition
- Enables ordered processing per property
- Consumer can maintain in-memory state per property

**Anomaly Detection in Stream:**
- Compare new price against 7-day rolling average
- Flag if change >80% drop or >500% increase
- Flagged prices go to review queue (don't serve immediately)
- This prevents supplier bugs from showing $1 rooms

**Cache Design (Redis):**
- Key: `price:{property_id}:{room_type_id}:{date}`
- Value: `{price_usd: 150.00, currency: "USD", supplier_id: 42, updated_at: ...}`
- TTL: 1 hour (safety net; CDC invalidation is primary)
- On cache miss: read from MySQL, populate cache, return

### Step 5: Operational Concerns

**Monitoring:**
- Price freshness: time between Kafka ingest and Redis write
- Anomaly rate: % of prices flagged as suspicious
- Cache hit rate: should be >95%
- Supplier health: updates per supplier per hour (detect supplier API failures)

**Failure modes:**
- Redis cluster down → fallback to MySQL (higher latency but works)
- Supplier sends garbage → schema validation + anomaly detection catches it
- Kafka consumer lag → auto-scale consumers, alert if lag >30 seconds

## Design 2: Clickstream Analytics Pipeline (1.8T Events/Day)

**Problem**: Design a pipeline to process 1.8 trillion clickstream events per day for user behavior analytics, A/B testing, and personalization at Agoda.

### Step 1: Requirements

```
Functional:
- Capture all user interactions (page views, clicks, searches, bookings)
- Real-time dashboards for ops team (within 5 minutes)
- Batch analytics for data scientists (daily aggregates)
- A/B test result computation
- Feed personalization models with user behavior features

Non-Functional:
- 1.8T events/day = ~20.8M events/sec
- Event size: ~500 bytes avg
- Throughput: ~10 GB/sec
- Daily raw storage: ~900 TB
- Retention: 90 days raw, 3 years aggregated
- Latency: <5 min for real-time dashboards, <6 hours for batch
```

### Architecture

```
┌─────────────────────────────────────────────────────────────────────┐
│  CLIENTS                                                           │
│  ┌──────┐  ┌──────┐  ┌──────┐                                     │
│  │ Web  │  │ iOS  │  │Android│                                     │
│  └──┬───┘  └──┬───┘  └──┬───┘                                     │
│     │         │         │                                          │
│     ▼         ▼         ▼                                          │
│  ┌──────────────────────────────────────┐                          │
│  │  Event Collection SDK                 │                          │
│  │  • Batching (send every 5s or 50 events)                        │
│  │  • Local queue (survive network failures)                       │
│  │  • Compression (gzip)                                           │
│  │  • Session ID + event sequence number                           │
│  └─────────────────┬────────────────────┘                          │
│                    │                                                │
│                    ▼                                                │
│  ┌──────────────────────────────────────┐                          │
│  │  Event Gateway (Load Balanced)        │                          │
│  │  • Validate schema                    │  Agoda's 2-Step         │
│  │  • Enrich (server timestamp, geo)     │  Logging: write         │
│  │  • Write to local disk (Step 1)       │  to disk, then          │
│  │  • Forwarder → Kafka (Step 2)         │  forward to Kafka       │
│  └─────────────────┬────────────────────┘                          │
│                    │                                                │
│                    ▼                                                │
│  ┌──────────────────────────────────────┐                          │
│  │  Kafka Analytics Cluster              │                          │
│  │  • 2000+ partitions                   │                          │
│  │  • Partitioned by user_id hash        │                          │
│  │  • Retention: 7 days                  │                          │
│  │  • Replication factor: 3              │                          │
│  │  • Compression: lz4                   │                          │
│  └────┬──────────────────┬───────────────┘                          │
│       │                  │                                          │
│       ▼                  ▼                                          │
│  ┌──────────┐    ┌──────────────┐                                  │
│  │ REAL-TIME │    │ BATCH PATH   │                                  │
│  │ PATH      │    │ (Hourly)     │                                  │
│  │           │    │              │                                  │
│  │ Flink     │    │ Spark Batch  │                                  │
│  │ • Session │    │ • Full ETL   │                                  │
│  │   windows │    │ • Sessionize │                                  │
│  │ • RT agg  │    │ • Aggregate  │                                  │
│  │ • Anomaly │    │ • Write to   │                                  │
│  │   detect  │    │   Delta Lake │                                  │
│  └────┬──────┘    └──────┬───────┘                                  │
│       │                  │                                          │
│       ▼                  ▼                                          │
│  ┌──────────┐    ┌──────────────┐                                  │
│  │ Apache   │    │ Delta Lake   │                                  │
│  │ Druid    │    │ (partitioned │                                  │
│  │ (RT OLAP)│    │  by hour)    │                                  │
│  └────┬──────┘    └──────┬───────┘                                  │
│       │                  │                                          │
│       ▼                  ▼                                          │
│  ┌──────────┐    ┌──────────────┐                                  │
│  │ Grafana  │    │ Presto/Trino │──▶ Data Scientists                │
│  │ RT Dash  │    │ SQL Access   │──▶ A/B Test Results              │
│  └──────────┘    └──────────────┘──▶ Feature Engineering           │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
```

### Key Design Decisions

**1. Why 2-Step Logging?**
- At 20.8M events/sec, direct Kafka writes from gateway would create massive connection overhead
- Local disk write: ~microseconds latency, never blocks the HTTP response
- Forwarder batches events, compresses, and sends efficiently
- If Kafka is temporarily down, events buffer on disk (hours of buffer capacity)

**2. Why Separate Real-Time and Batch Paths? (Lambda)**
- Real-time: Flink for minute-level aggregates (dashboard, alerting)
- Batch: Spark for exact aggregates (A/B test results need precision)
- Batch corrects any approximations from streaming (late events, watermark drops)

**3. Sessionization Challenge**
- Group events by user into sessions (30-min inactivity gap)
- In streaming: Flink session windows with 30-min gap
- In batch: Sort by user_id + timestamp, use lag/lead to detect gaps
- Late events: watermark allows 5-min late arrival; anything later handled in batch

**4. Storage Optimization**
- 900 TB/day raw → need aggressive compression and lifecycle management
- Parquet + LZ4 compression: ~5x compression ratio → ~180 TB/day
- 90-day retention: ~16 PB raw storage
- Tiered storage: hot (7 days, SSD), warm (30 days, HDD), cold (90 days, S3 Glacier)

## Design 3: Hotel Search Ranking Feature Pipeline

**Problem**: Design the data pipeline that powers Agoda's hotel search ranking, from raw data to model features served at 3.5M entities/sec with <10ms P99.

### Architecture

```
┌─────────────────────────────────────────────────────────────────────────┐
│                  SEARCH RANKING FEATURE PIPELINE                       │
├─────────────────────────────────────────────────────────────────────────┤
│                                                                         │
│  DATA SOURCES                                                           │
│  ┌──────────┐ ┌──────────┐ ┌──────────┐ ┌──────────┐ ┌──────────┐    │
│  │Bookings  │ │ Reviews  │ │Clickstrm │ │ Prices   │ │ Property │    │
│  │ (MySQL)  │ │ (MySQL)  │ │ (Kafka)  │ │ (Kafka)  │ │ (MySQL)  │    │
│  └────┬─────┘ └────┬─────┘ └────┬─────┘ └────┬─────┘ └────┬─────┘    │
│       │             │            │             │            │           │
│  ═════╪═════════════╪════════════╪═════════════╪════════════╪═══════   │
│       │ OFFLINE PATH (Batch - Daily)           │            │           │
│       ▼             ▼            │             │            ▼           │
│  ┌─────────────────────────────┐ │             │  ┌──────────────┐     │
│  │ Spark Batch Jobs            │ │             │  │ Spark Batch  │     │
│  │ • 30-day booking features   │ │             │  │ • Static     │     │
│  │ • Review features           │ │             │  │   features   │     │
│  │ • Conversion features       │ │             │  │ • Enrichment │     │
│  └──────────┬──────────────────┘ │             │  └──────┬───────┘     │
│             │                     │             │         │             │
│             ▼                     │             │         ▼             │
│  ┌─────────────────────────────────────────────────────────────────┐   │
│  │                     FEATURE TABLE (Hive/Delta Lake)             │   │
│  │  property_id | bookings_30d | avg_review | ctr | cvr | ...     │   │
│  └───────────────────────────┬─────────────────────────────────────┘   │
│                              │                                         │
│                              │ Daily load to online store              │
│                              ▼                                         │
│  ═══════════════════════════════════════════════════════════════════   │
│       │ ONLINE PATH (Real-Time)    │            │                      │
│       │                            ▼            ▼                      │
│       │                   ┌──────────────────────────┐                │
│       │                   │ Flink Streaming Jobs     │                │
│       │                   │ • Real-time CTR per prop │                │
│       │                   │ • Current min price      │                │
│       │                   │ • Live availability      │                │
│       │                   └───────────┬──────────────┘                │
│       │                               │                               │
│       ▼                               ▼                               │
│  ┌──────────────────────────────────────────────────────────────┐     │
│  │                    ScyllaDB (Source of Truth)                 │     │
│  │  property_id → {all features: batch + real-time}             │     │
│  └─────────────────────────┬────────────────────────────────────┘     │
│                            │                                          │
│                            ▼                                          │
│  ┌──────────────────────────────────────────────────────────────┐     │
│  │               DragonflyDB (Centralized Cache)                │     │
│  │               P99 < 3ms | Millions of QPS                    │     │
│  └─────────────────────────┬────────────────────────────────────┘     │
│                            │                                          │
│                            ▼                                          │
│  ┌──────────────────────────────────────────────────────────────┐     │
│  │         Feature Serving Layer (Rust on Kubernetes)            │     │
│  │         3.5M entities/sec | P99 < 10ms SLA                   │     │
│  │  ┌──────────────┐  ┌────────────────┐  ┌────────────────┐   │     │
│  │  │ Scala SDK    │  │ Python SDK     │  │  REST API      │   │     │
│  │  │ (client cache│  │ (client cache  │  │  (for non-SDK  │   │     │
│  │  │  ~50% hits)  │  │  for training) │  │   consumers)   │   │     │
│  │  └──────────────┘  └────────────────┘  └────────────────┘   │     │
│  └──────────────────────────────────────────────────────────────┘     │
│                            │                                          │
│                            ▼                                          │
│  ┌──────────────────────────────────────────────────────────────┐     │
│  │              CONSUMERS                                        │     │
│  │  ┌──────────────┐  ┌────────────────┐  ┌────────────────┐   │     │
│  │  │ Search       │  │ Pricing        │  │ Recommendation │   │     │
│  │  │ Ranking      │  │ Model          │  │ Engine         │   │     │
│  │  │ Model        │  │                │  │                │   │     │
│  │  └──────────────┘  └────────────────┘  └────────────────┘   │     │
│  └──────────────────────────────────────────────────────────────┘     │
└─────────────────────────────────────────────────────────────────────────┘
```

### Key Design Decisions

**1. Why ScyllaDB as Source of Truth?**
- CQL-compatible (familiar Cassandra interface)
- Consistent low-latency at scale (shared-nothing architecture)
- Handles mixed read/write workloads well
- Automatic data distribution across nodes (vnodes/tablets)

**2. Why DragonflyDB Instead of Redis?**
- Multi-threaded (Redis is single-threaded per instance)
- No GC pauses (C++ vs. no GC needed comparison)
- Higher memory efficiency (up to 60% less memory than Redis for same data)
- Drop-in Redis replacement (same protocol)
- Agoda benchmarked P99 <3ms on 64-core AMD EPYC processors

**3. Why Rust for Serving Layer?**
- No garbage collection → predictable latency (no GC pauses)
- Memory safety without runtime overhead
- Excellent performance for I/O-heavy workloads
- Efficient resource usage → fewer pods needed in Kubernetes

**4. Offline-Online Feature Consistency**
- Training features computed by Spark batch (offline)
- Serving features read from ScyllaDB (online)
- Both paths read from same source data (Delta Lake)
- Nightly validation job: compare offline features vs online store
- Alert if >1% divergence → indicates pipeline bug

# Part 11: Common Mistakes, Tricks & Edge Cases

## 11.1 Common Interview Mistakes

| Mistake | Why It Hurts | What to Do Instead |
|---------|-------------|--------------------|
| Jumping straight into solution | Shows lack of engineering rigor | Always clarify requirements first (3-5 min) |
| Naming tools without justification | Sounds like buzzword dropping | "I'd use Kafka because we need ordered, replayable events with multiple consumers" |
| Ignoring failure modes | System won't survive production | For every component, ask "what if this fails?" |
| Over-engineering for day 1 | Unnecessary complexity | Start simple, explain how it scales ("Phase 1: single node Redis; Phase 2: Redis Cluster") |
| Under-engineering | Won't handle the stated scale | Do back-of-envelope math to prove your design handles the load |
| Forgetting data quality | Bad data → bad decisions | Always include validation, monitoring, and alerting |
| Not discussing trade-offs | Shows shallow thinking | Every decision has pros/cons — state them explicitly |
| Monolithic thinking | Everything in one service | Split by domain: ingestion, processing, storage, serving |
| Ignoring operational aspects | System is a black box | Cover monitoring, alerting, debugging, on-call experience |
| Not adapting to feedback | Misses collaborative signal | Listen to interviewer's hints; they're guiding you |

## 11.2 Tricks & Pro Tips

### Trick 1: The "Three Whys" for Every Decision
```
"I chose Kafka for the messaging layer."
→ Why Kafka specifically?
  "Because we need ordered events per key, replay capability for backfills, 
   and multiple consumer groups (real-time + batch)."
→ Why not a simpler queue like SQS?
  "SQS doesn't support ordered consumption or replay, and we need multiple 
   consumers reading the same stream independently."
→ Why not Pulsar or Kinesis?
  "Kafka has the largest ecosystem, Agoda already uses it at scale, and 
   the operational expertise exists. Pulsar would be a valid alternative 
   if we needed built-in tiered storage."
```

### Trick 2: Start with the Query Pattern
```
Before designing storage, ask:
"What queries will we run against this data?"

Query: "Find all bookings for property X in January"
→ Index on (property_id, check_in_date)
→ Partition data lake by check_in_date

Query: "Get feature vector for property X in <10ms"
→ Key-value store (ScyllaDB) with property_id as partition key
→ Cache in front (DragonflyDB)

Query: "Daily revenue by city for last 30 days"
→ Pre-aggregated Gold table, partitioned by date
→ Or OLAP engine (Druid) for ad-hoc
```

### Trick 3: The "Onion" Approach for Deep Dives
```
Layer 1: High-level components (boxes and arrows)
Layer 2: Data flow and formats (Avro, Parquet, JSON)
Layer 3: Key algorithms (dedup, windowing, aggregation)
Layer 4: Failure handling (retry, DLQ, circuit breaker)
Layer 5: Operational (monitoring, alerting, runbooks)
```

## 11.3 Edge Cases & Challenges at Scale

### Edge Case 1: Late-Arriving Events
```
Problem: User books a hotel, but the booking event arrives 3 hours late
         due to network issue on their device.

Impact:
- Real-time dashboard showed incorrect booking count
- Streaming aggregation already computed and closed the window
- Feature store has stale conversion features for that property

Solutions:
1. Watermark in streaming: Allow configurable late arrival (e.g., 1 hour)
2. Batch reconciliation: Nightly batch job corrects streaming results
3. Lambda architecture: Batch result is the "source of truth"
4. Delta Lake MERGE: Late events upsert into existing data
5. Allowed lateness in Flink: Process late events into a side output
```

### Edge Case 2: Schema Evolution
```
Problem: Upstream team adds a "loyalty_points" field to booking events.
         Your pipeline breaks because the schema doesn't match.

Solutions:
1. Schema Registry (Avro/Protobuf) with compatibility modes:
   - BACKWARD: New schema can read old data (add fields with defaults)
   - FORWARD: Old schema can read new data (don't remove/rename fields)
   - FULL: Both backward and forward compatible
2. Set compatibility to BACKWARD_TRANSITIVE for all topics
3. Data contract: Define SLA with upstream team:
   - No breaking changes without 2-week notice
   - New fields must have default values
   - Removed fields must be deprecated first
```

### Edge Case 3: Data Skew in City-Level Aggregations
```
Problem: Bangkok has 50,000 properties. Small island resort has 5.
         Spark groupBy("city") → Bangkok partition takes 100x longer.

Impact: 1 task takes 2 hours; 999 tasks finish in 1 minute.
        Total job time = 2 hours (limited by slowest task).

Solutions:
1. Salting: Add random prefix to city key
   - bangkok_0, bangkok_1, ..., bangkok_9
   - Aggregate per salt, then aggregate again across salts
2. AQE (Spark 3.x): Automatically detects and splits skewed partitions
3. Two-phase aggregation: Local pre-aggregation before shuffle
4. Custom partitioner: Assign top cities to multiple partitions
```

### Edge Case 4: Thundering Herd on Cache
```
Problem: DragonflyDB cache entry for popular property expires.
         10,000 concurrent requests hit ScyllaDB simultaneously.

Impact: ScyllaDB overloaded, cascading failures.

Solutions:
1. Cache warming: Proactively refresh before TTL expires
2. Lock/Singleflight: Only one request fetches from DB;
   others wait for the result
3. Stale-while-revalidate: Serve stale data while refreshing in background
4. Jittered TTL: Add random jitter to prevent synchronized expiration
   TTL = base_ttl + random(0, 0.2 * base_ttl)
```

### Edge Case 5: Duplicate Events from Kafka
```
Problem: Producer retries after timeout → same event in Kafka twice.
         Consumer processes both → double-counted booking.

Impact: Revenue reports inflated. Overbooking risk.

Solutions:
1. Idempotent producer: enable.idempotence=true
   → Kafka deduplicates within producer session
2. Consumer-side dedup: Track processed event IDs in state store
3. Spark dedup: Window function to keep latest per primary key
4. Delta Lake MERGE: Upsert by booking_id → naturally deduplicated
5. Exactly-once Kafka transactions (for Kafka-to-Kafka pipelines)
```

### Edge Case 6: Timezone Handling in Global OTA
```
Problem: User in New York books a hotel in Bangkok.
         - User's timezone: UTC-5
         - Hotel's timezone: UTC+7
         - Server timezone: UTC
         Which timezone for check-in date? For event_timestamp?

Solution:
1. ALWAYS store event_timestamp in UTC
2. Check-in/check-out dates: Store as DATE (no timezone) in hotel's local timezone
3. Analytics: Convert to UTC for global aggregations
4. Display: Convert to user's timezone in the UI layer
5. Partitioning: Use UTC date for data lake partitioning
```

# Part 12: Mock Interview Q&A Bank

## 12.1 Warm-Up Questions (5 min each)

---

**Q1: Explain the difference between a data lake, data warehouse, and lakehouse.**

**A:**
| Aspect | Data Lake | Data Warehouse | Lakehouse |
|--------|-----------|---------------|----------|
| Storage | Raw files (S3/HDFS) | Structured tables | Files + table format |
| Schema | Schema-on-read | Schema-on-write | Schema enforcement optional |
| Format | Any (JSON, CSV, Parquet) | Proprietary/optimized | Open (Delta, Iceberg) |
| ACID | No | Yes | Yes |
| Cost | Low | High | Low |
| Query | Spark/Presto | SQL-native | Both |
| Use case | ML, raw data storage | BI, reporting | Both ML and BI |
| Example | HDFS + Spark | Snowflake, BigQuery | Delta Lake, Apache Iceberg |

**Agoda context**: Uses a lakehouse approach with Delta Lake on HDFS/S3 for the data lake, with Hive/Presto for SQL access.

---

**Q2: How would you handle a Spark job that's running for 6 hours instead of the expected 30 minutes?**

**A: Debugging Steps (in order):**
1. **Check Spark UI → Stages tab**: Identify the slow stage
2. **Check Task Distribution**: If 1 task is 100x slower → data skew
3. **Check Shuffle Metrics**: Large shuffle → too many/few partitions
4. **Check GC Time**: If >10% of task time → memory pressure
5. **Check Executor Logs**: OOM errors, serialization issues

**Common Fixes:**
- Data skew → Enable AQE or salt the skewed key
- Too many small files → Compact input files first
- Cartesian join → Missing join condition (check the query plan)
- Python UDFs → Replace with Spark SQL or Pandas UDFs (10-100x faster)
- Wrong file format → Switch from CSV/JSON to Parquet

---

**Q3: What's the difference between at-most-once, at-least-once, and exactly-once delivery?**

**A:**
| Guarantee | Behavior | Use Case |
|-----------|----------|----------|
| At-most-once | Fire and forget; may lose messages | Metrics, logging (acceptable loss) |
| At-least-once | Retry on failure; may duplicate | Default for most pipelines (dedup downstream) |
| Exactly-once | Each message processed exactly once | Financial transactions, booking counts |

**How to achieve exactly-once end-to-end:**
1. Idempotent Kafka producer (`enable.idempotence=true`)
2. Transactional Kafka (for Kafka-to-Kafka)
3. Consumer checkpoint + idempotent sink (Delta Lake MERGE)
4. Deduplication at processing layer (window function on primary key)

---

## 12.2 Medium-Depth Questions (10 min each)

---

**Q4: How would you design a data pipeline to detect fraudulent hotel bookings in real-time?**

**A:**
```
Architecture:
Booking Events (Kafka) → Flink Streaming → ML Model → Action

Feature computation (in Flink):
- User features: booking frequency, avg spend, new account flag
- Session features: number of searches before booking, time-to-book
- Payment features: new card, card-country mismatch, multiple failed attempts
- Device features: VPN/proxy, device fingerprint, unusual user-agent

Real-time features (Flink state):
- Bookings in last 1 hour for this user
- Bookings in last 1 hour for this credit card
- Bookings in last 1 hour from this IP

Enrichment:
- Join with user profile (from feature store)
- Join with property fraud history
- Join with IP reputation database

Model:
- Lightweight model (logistic regression/XGBoost) for real-time
- P99 inference <5ms
- Score: 0-100 (probability of fraud)

Action:
- Score < 30: Auto-approve
- Score 30-70: Flag for manual review, hold booking
- Score > 70: Auto-reject, alert fraud team

Feedback loop:
- Manual review outcomes → training data for model improvement
- Retrain model weekly with new labeled data
```

---

**Q5: You discover that your data pipeline has been producing incorrect results for the last 2 weeks. How do you handle this?**

**A (Incident Response):**
```
Phase 1: Triage (first 30 min)
1. Assess blast radius: What downstream systems are affected?
2. Decide: Stop the pipeline? Or let it run while investigating?
3. Communicate: Alert stakeholders (data consumers, product team)

Phase 2: Root Cause (1-4 hours)
1. Identify when the bug was introduced (git blame, deployment log)
2. Understand the bug (incorrect join, wrong filter, schema change)
3. Assess data impact (which records are affected, how badly)

Phase 3: Fix & Backfill (hours-days)
1. Fix the bug in code
2. Test with a sample of data
3. Backfill from Bronze layer (this is why we keep raw data!)
   - Read from Bronze (raw, immutable, correct)
   - Process with fixed logic
   - Write to Silver using Delta Lake MERGE (upsert)
   - Propagate to Gold + Feature Store
4. Validate: Compare backfilled results with original

Phase 4: Prevention
1. Add data quality check that would have caught this
2. Add unit/integration test for the fixed code
3. Post-mortem document with timeline and action items
4. Consider: Should the pipeline auto-halt when quality checks fail?
```

---

**Q6: How would you migrate a batch pipeline (runs every 6 hours) to near-real-time (5-minute latency)?**

**A:**
```
Phase 1: Dual-run (parallel paths)
  ┌─ Existing batch (6-hour) → Current consumers (unchanged)
  │
  └─ New streaming pipeline → Shadow output (for validation only)

Phase 2: Validate
  - Compare batch output vs streaming output daily
  - Check for correctness, completeness, latency
  - Run for 2-4 weeks

Phase 3: Gradual cutover
  - Route 10% of consumers to streaming output
  - Monitor error rates and data quality
  - Increase to 50%, then 100%

Phase 4: Deprecate batch
  - Keep batch as fallback for 30 days
  - Then decommission

Key technical changes:
1. Source: Batch reads from data lake → Streaming reads from Kafka
2. Processing: Spark batch → Spark Structured Streaming or Flink
3. State: Batch recomputes from scratch → Streaming maintains state
4. Sink: Overwrite → Append/Upsert
5. Error handling: Retry whole batch → DLQ per event + retry
6. Monitoring: Run completion check → Consumer lag + latency check
```

---

## 12.3 Senior-Level Deep Dive Questions (15-20 min each)

---

**Q7: Design a system to detect and handle data pipeline SLA breaches automatically.**

**A:**
```
Components:

1. SLA Definition Service
   - Each pipeline registers its SLA:
     {pipeline: "booking_etl", max_latency: "2h", min_completeness: 0.99, 
      freshness: "6h", owner: "data-platform@agoda.com"}

2. Pipeline Metadata Collector
   - Tracks: last_successful_run, records_processed, runtime_seconds
   - Source: Airflow metadata, Spark event logs, custom pipeline events

3. SLA Monitor (Flink/cron job)
   - Every minute: check each pipeline against its SLA
   - Conditions:
     a) Freshness breach: current_time - last_success > max_freshness
     b) Latency breach: last_runtime > max_latency
     c) Completeness breach: records_processed < threshold
     d) Quality breach: quality_score < min_quality

4. Alert & Action Engine
   - Level 1 (warning): Slack notification to owner
   - Level 2 (breach): PagerDuty alert, auto-create Jira ticket
   - Level 3 (critical): Auto-restart pipeline, escalate to manager

5. Auto-Remediation
   - Spark OOM → auto-increase executor memory and retry
   - Kafka lag → auto-scale consumer group
   - Data quality failure → route to DLQ, process clean data
   - Source unavailable → exponential backoff retry

6. Dashboard
   - Real-time SLA status for all pipelines
   - Historical SLA compliance (target: 99.9%)
   - Time-to-resolution metrics
```

---

**Q8: How do you ensure data consistency between your OLTP database (MySQL), data lake, and feature store?**

**A:**
```
The Consistency Triangle:

          MySQL (Source of Truth for OLTP)
            │
            │ CDC (Debezium)
            ▼
          Kafka (Event Log)
          ╱         ╲
    Spark Batch    Flink Streaming
         │              │
         ▼              ▼
    Delta Lake     Feature Store
    (Analytics)    (Online Serving)

Consistency Mechanisms:

1. CDC ensures data lake reflects MySQL changes
   - Debezium captures INSERT, UPDATE, DELETE from binlog
   - Preserves order per primary key (Kafka partition by PK)

2. Delta Lake ensures ACID on the data lake
   - MERGE operations are idempotent (safe to replay)
   - Transaction log ensures atomicity

3. Feature Store consistency
   - Streaming path updates features within seconds
   - Daily batch job performs full reconciliation:
     a) Read all features from Feature Store
     b) Recompute all features from Delta Lake
     c) Compare and overwrite any mismatches
     d) Alert if mismatch rate > 1%

4. Cross-system validation (nightly)
   - Row count: MySQL table count vs Delta Lake count vs Feature Store count
   - Checksum: Hash of key columns should match across systems
   - Sample verification: Random sample of 1000 records, verify field-by-field

5. Handling eventual consistency
   - Accept that systems may be inconsistent for seconds-minutes
   - Design consumers to handle temporary inconsistency
   - Feature Store: client SDK caches → may serve slightly stale data
   - This is acceptable for search ranking (not for booking/payment)
```

---

**Q9: Walk me through how you would design Agoda's Featureflow system from scratch.**

**A:**
```
Goal: Allow anyone (not just ML engineers) to create, test, and deploy
      ML features using just SQL.

Architecture:

┌────────────────────────────┐
│  Featureflow UI            │
│  • SQL editor              │
│  • Feature set management  │
│  • Performance dashboard   │
└────────────┬───────────────┘
             │
             ▼
┌────────────────────────────┐
│  Feature Registry (MySQL)  │
│  • Feature definitions     │
│  • SQL queries             │
│  • Default values          │
│  • Entity types            │
│  • Versioning              │
└────────────┬───────────────┘
             │
             ▼
┌────────────────────────────┐
│  Orchestrator (Airflow)    │
│  • Schedule feature jobs   │
│  • Dependency management   │
│  • Smart feature selection │
│    (prioritize promising)  │
└────────────┬───────────────┘
             │
    ┌────────┴────────┐
    ▼                 ▼
┌──────────┐   ┌──────────────┐
│ Babyship │   │ Babyship     │  (One per feature set)
│ (Spark)  │   │ (Spark)      │
│ Feature  │   │ Feature      │
│ compute  │   │ compute      │
└────┬─────┘   └──────┬───────┘
     │                │
     ▼                ▼
┌──────────────────────────────┐
│  Feature Data (Delta Lake)   │
│  • Historical feature values │
│  • Joined with labels        │
└────────────┬─────────────────┘
             │
             ▼
┌────────────────────────────┐
│  Watermelon (Python)       │
│  • Load processed data     │
│  • Train model (sklearn)   │
│  • Compute statistics      │
│  • Submit to model store   │
└────────────┬───────────────┘
             │
             ▼
┌────────────────────────────┐
│  Experiment Platform       │
│  • Sandbox testing         │
│  • A/B test deployment     │
│  • Result analysis         │
└────────────────────────────┘

Key Design Decisions:
1. SQL as interface → Low barrier to entry (no Python/Spark knowledge needed)
2. Smart Feature Selection → Don't recompute all features; prioritize based on
   recent performance (saves compute)
3. Babyship per feature set → Isolation; one bad SQL doesn't kill others
4. Separate training container (Watermelon) → Python flexibility for ML
5. Sandbox environment → Safe testing before production deployment
```

## 12.4 Rapid-Fire Questions (2-3 min each)

**Q10: Parquet vs ORC vs Avro — when to use which?**

| Format | Type | Best For | Compression | Schema Evolution |
|--------|------|----------|-------------|------------------|
| Parquet | Columnar | Analytics (SELECT specific columns) | Excellent | Moderate |
| ORC | Columnar | Hive-heavy workloads | Excellent | Moderate |
| Avro | Row-based | Kafka messages, full-row reads | Good | Excellent |

**Rule**: Avro for streaming/Kafka, Parquet for data lake/analytics.

---

**Q11: What is a slowly changing dimension (SCD) and how do you handle it?**

In OTA context: Hotel star rating changes from 4 to 5.

| SCD Type | Strategy | Example |
|----------|----------|--------|
| Type 1 | Overwrite | Update star_rating = 5 (lose history) |
| Type 2 | Add new row with validity dates | Row 1: 4 stars (2020-2024), Row 2: 5 stars (2024-now) |
| Type 3 | Add column | current_star_rating=5, previous_star_rating=4 |

**For Agoda**: Use SCD Type 2 for property attributes (need historical accuracy for analytics).

**Delta Lake implementation:**
```sql
MERGE INTO dim_property target
USING staging source
ON target.property_id = source.property_id AND target.is_current = true
WHEN MATCHED AND target.star_rating != source.star_rating THEN
  UPDATE SET is_current = false, valid_to = current_date()
WHEN NOT MATCHED THEN
  INSERT (property_id, star_rating, is_current, valid_from, valid_to)
  VALUES (source.property_id, source.star_rating, true, current_date(), '9999-12-31')
```

---

**Q12: Explain the CAP theorem with a real OTA example.**

**CAP**: In a distributed system, you can only guarantee 2 of 3:
- **Consistency**: Every read gets the most recent write
- **Availability**: Every request gets a response
- **Partition Tolerance**: System works despite network partitions

**OTA Examples:**
- **Booking system (CP)**: If network partition occurs, better to reject bookings than risk double-booking. Sacrifice availability for consistency.
- **Search results (AP)**: If data center is partitioned, show slightly stale results rather than error page. Sacrifice consistency for availability.

---

**Q13: What is backpressure and how do you handle it?**

**Backpressure**: When a downstream system can't keep up with the rate of incoming data.

**Example**: Kafka consumer processing takes 100ms per message, but messages arrive every 10ms.

**Handling strategies:**
1. **Buffer**: Kafka itself acts as buffer (configurable retention)
2. **Scale consumers**: Add more consumers (up to partition count)
3. **Rate limit producers**: Throttle upstream if needed
4. **Batch processing**: Process messages in batches, not one-by-one
5. **Shed load**: Drop low-priority events during overload (e.g., debug logs)
6. **Spark Structured Streaming**: `maxOffsetsPerTrigger` limits records per batch

---

**Q14: How do you handle idempotency in a distributed data pipeline?**

**Definition**: An operation is idempotent if executing it multiple times has the same effect as executing it once.

**Why it matters**: In distributed systems, components retry on failure. Without idempotency, retries cause duplicates.

**Implementation levels:**
1. **Producer**: Kafka idempotent producer (dedup by sequence number)
2. **Pipeline**: Process event, check if result already exists (UPSERT/MERGE)
3. **Sink**: Use natural keys (booking_id) as primary key → INSERT is idempotent
4. **API**: Idempotency key in request header → server deduplicates

---

**Q15: What is the difference between a message queue (SQS) and an event log (Kafka)?**

| Feature | Message Queue (SQS) | Event Log (Kafka) |
|---------|--------------------|-----------------|
| Consumption | Message deleted after processing | Messages retained, offset-based |
| Consumers | Single consumer per message | Multiple consumers read independently |
| Replay | No (once consumed, gone) | Yes (seek to any offset) |
| Ordering | Not guaranteed (FIFO available) | Guaranteed within partition |
| Use case | Task queues, decoupling | Event sourcing, data pipelines, analytics |

**For data engineering**: Almost always Kafka (replay for backfill, multiple consumers).

# Part 13: Interview Day Cheat Sheet

## Quick Reference Card

### When You Hear... → Think About...

| Interviewer Says | Your Framework |
|-----------------|---------------|
| "Design a data pipeline" | Sources → Ingestion (Kafka) → Processing (Spark) → Storage (Delta) → Serving |
| "Make it real-time" | Spark Structured Streaming or Flink, watermarks, state management |
| "Handle high throughput" | Kafka partitioning, consumer groups, Spark parallelism |
| "Ensure data quality" | Validation checks, anomaly detection, DLQ, monitoring |
| "Handle failures" | Retries, DLQ, checkpointing, idempotent writes, backfill strategy |
| "Scale to 10x" | Horizontal scaling, sharding, caching, async processing |
| "Prevent duplicates" | Idempotent producer, dedup in processing, MERGE in storage |
| "Handle late data" | Watermarks, batch reconciliation, Lambda architecture |
| "Schema changes" | Schema Registry, backward compatibility, data contracts |
| "Monitor this system" | 4 pillars: Metrics, Logs, Traces, Alerts |
| "What if the whole DC goes down?" | Active-active, Kafka MirrorMaker, cross-DC replication |
| "How to test this pipeline?" | Unit tests (transformations), integration tests (end-to-end), data quality tests |

### Numbers to Remember (Agoda Scale)

```
1.8 trillion Kafka events/day → ~20.8M events/sec
3.5M feature store reads/sec → P99 <10ms
2M+ hotel properties worldwide
50x feature store scaling in 2 years
10,000+ ML features processed through Featureflow
~30 people manage the entire data infrastructure
```

### Communication Structure

```
1. State the approach: "I'll use X because..."
2. Explain the trade-off: "The alternative is Y, but X is better here because..."
3. Address the risk: "The main risk with X is... and I'd mitigate it by..."
4. Connect to requirements: "This meets our SLA of... because..."
```

### Red Flags to Avoid

```
✗ "We can just use a single machine" (at Agoda scale, almost never true)
✗ "We'll figure out monitoring later" (operational excellence is Day 1)
✗ "This will never fail" (everything fails; show you've planned for it)
✗ "We need microservices for everything" (over-engineering signal)
✗ "I'll use [technology] because it's popular" (justify with requirements)
```

---

## Good Luck! Remember:

1. **Clarify first** — 3-5 minutes of questions shows seniority
2. **Estimate** — Math proves your design works
3. **Trade-offs** — Every decision has pros and cons; state them
4. **Think operationally** — How will you debug this at 3 AM?
5. **Collaborate** — Listen to the interviewer's hints and adapt
6. **Stay structured** — Use the 5-step framework consistently